# Derivatives Volatility Options Fundamentals

This consolidated Python workbook brings the related chapters together in a single guided sequence.

## Chapters

1. Options Basics and Terminology
2. Option Trading Strategies

Work through the chapters in order, or use the headings to jump directly to the topic you need.


## Chapter 1: Options Basics and Terminology

**Level:** Beginner\n**Concepts Covered:** 6

---

## Overview

This comprehensive notebook covers options basics and terminology with detailed explanations, mathematical derivations, Python implementations, and practical examples.

### 1. Introduction

Options are financial derivatives that give the holder the right, but not the obligation, to buy or sell an underlying asset at a predetermined price (strike price) on or before a specified date (expiration date). Unlike futures contracts, options provide asymmetric payoffs - limited downside for buyers and potentially unlimited upside.

#### Real-World Applications

- **Hedging**: Portfolio managers use protective puts to insure against market downturns
- **Income Generation**: Covered call strategies generate premium income on existing stock positions
- **Speculation**: Traders leverage options for directional bets with defined risk
- **Corporate Finance**: Executive stock options align management incentives with shareholders

#### Learning Objectives

By the end of this notebook, you will be able to:

1. **Distinguish** between call and put options and explain their fundamental characteristics
2. **Calculate** option payoffs and profits at expiration for both long and short positions
3. **Classify** options by moneyness (ITM, ATM, OTM) and understand their practical implications
4. **Construct** payoff diagrams for basic option positions
5. **Compare** European vs American options and their exercise features

#### Prerequisites

- Basic understanding of financial markets and stock trading
- Familiarity with Python, NumPy, and Matplotlib
- Elementary probability concepts

#### Industry Context

The global options market has grown exponentially since the Chicago Board Options Exchange (CBOE) opened in 1973. Today, millions of option contracts trade daily across equities, indices, currencies, and commodities. Understanding options is essential for:

- Risk managers in banks and hedge funds
- Equity traders and market makers
- Financial engineers developing structured products
- Individual investors seeking portfolio protection

**Estimated Time**: 90-120 minutes

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats, optimize
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('[OK] Libraries imported successfully')

### 2. Call and Put Options

#### Theory

A **call option** gives the holder the right to **buy** the underlying asset at the strike price. A **put option** gives the holder the right to **sell** the underlying asset at the strike price.

##### Call Option Characteristics

- **Buyer's Perspective**: Profits when the underlying price rises above the strike price
- **Maximum Loss**: Limited to the premium paid
- **Maximum Gain**: Theoretically unlimited as the underlying price can rise indefinitely
- **Typical Use**: Bullish outlook, leverage, or hedging short positions

##### Put Option Characteristics

- **Buyer's Perspective**: Profits when the underlying price falls below the strike price
- **Maximum Loss**: Limited to the premium paid
- **Maximum Gain**: Limited to strike price minus premium (underlying can't go below zero)
- **Typical Use**: Bearish outlook, portfolio protection, or hedging long positions

#### Key Terminology

- **Premium**: The price paid to purchase the option
- **Strike Price (K)**: The predetermined price at which the option can be exercised
- **Underlying Price (S)**: The current market price of the asset
- **Expiration Date (T)**: The last date the option can be exercised
- **Intrinsic Value**: The immediate exercise value of the option
- **Time Value**: Premium minus intrinsic value; represents optionality

#### The Options Contract

An options contract specifies:
- Underlying asset (e.g., 100 shares of AAPL)
- Strike price (e.g., $150)
- Expiration date (e.g., third Friday of the month)
- Option type (Call or Put)
- Exercise style (European or American)

#### Mathematical Formulation

The **payoff** at expiration for options is defined as:

**Call Option Payoff (Long Position):**
$$
\text{Payoff}_{\text{call}} = \max(S_T - K, 0)
$$

**Call Option Profit:**
$$
\text{Profit}_{\text{call}} = \max(S_T - K, 0) - \text{Premium}
$$

**Put Option Payoff (Long Position):**
$$
\text{Payoff}_{\text{put}} = \max(K - S_T, 0)
$$

**Put Option Profit:**
$$
\text{Profit}_{\text{put}} = \max(K - S_T, 0) - \text{Premium}
$$

where:
- $S_T$ = underlying price at expiration
- $K$ = strike price
- $\max(a, b)$ = maximum of $a$ and $b$

The $\max$ function ensures the option is only exercised when profitable. If exercising would result in a loss, the option expires worthless and the holder simply loses the premium paid.

**Intrinsic Value:**
$$
\text{Intrinsic Value}_{\text{call}} = \max(S_0 - K, 0)
$$
$$
\text{Intrinsic Value}_{\text{put}} = \max(K - S_0, 0)
$$

where $S_0$ is the current underlying price.

In [ ]:
# Python implementation for Call and Put Options

def call_payoff(S_T, K):
    """
    Calculate call option payoff at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
        
    Returns:
    --------
    float or np.array
        Payoff value (non-negative)
    """
    return np.maximum(S_T - K, 0)

def call_profit(S_T, K, premium):
    """
    Calculate call option profit at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
    premium : float
        Option premium paid
        
    Returns:
    --------
    float or np.array
        Profit (can be negative)
    """
    return call_payoff(S_T, K) - premium

def put_payoff(S_T, K):
    """
    Calculate put option payoff at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
        
    Returns:
    --------
    float or np.array
        Payoff value (non-negative)
    """
    return np.maximum(K - S_T, 0)

def put_profit(S_T, K, premium):
    """
    Calculate put option profit at expiration.
    
    Parameters:
    -----------
    S_T : float or np.array
        Underlying price at expiration
    K : float
        Strike price
    premium : float
        Option premium paid
        
    Returns:
    --------
    float or np.array
        Profit (can be negative)
    """
    return put_payoff(S_T, K) - premium

# Example calculation
K = 100  # Strike price
S_T_example = 110  # Underlying at expiration
call_premium = 5
put_premium = 3

print("=" * 60)
print("EXAMPLE: Option Payoffs and Profits")
print("=" * 60)
print(f"Strike Price (K): ${K}")
print(f"Underlying at Expiration (S_T): ${S_T_example}")
print(f"Call Premium: ${call_premium}")
print(f"Put Premium: ${put_premium}")
print("-" * 60)
print(f"Call Payoff: ${call_payoff(S_T_example, K):.2f}")
print(f"Call Profit: ${call_profit(S_T_example, K, call_premium):.2f}")
print(f"Put Payoff: ${put_payoff(S_T_example, K):.2f}")
print(f"Put Profit: ${put_profit(S_T_example, K, put_premium):.2f}")
print("=" * 60)

print('\n[OK] Call and Put Options implementation complete')

In [ ]:
# Visualization for Call and Put Options Payoff Diagrams

# Define parameters
K = 100  # Strike price
call_premium = 5
put_premium = 3

# Create range of underlying prices
S_range = np.linspace(70, 130, 200)

# Calculate payoffs and profits
call_payoffs = call_payoff(S_range, K)
call_profits = call_profit(S_range, K, call_premium)
put_payoffs = put_payoff(S_range, K)
put_profits = put_profit(S_range, K, put_premium)

# Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Call Payoff
axes[0, 0].plot(S_range, call_payoffs, linewidth=2.5, color='#2E86AB', label='Call Payoff')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[0, 0].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[0, 0].fill_between(S_range, 0, call_payoffs, alpha=0.2, color='#2E86AB')
axes[0, 0].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[0, 0].set_ylabel('Payoff ($)', fontsize=11)
axes[0, 0].set_title('Long Call Payoff', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper left', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)

# Call Profit
axes[0, 1].plot(S_range, call_profits, linewidth=2.5, color='#A23B72', label='Call Profit')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[0, 1].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[0, 1].axvline(x=K+call_premium, color='green', linestyle='--', linewidth=1.5, alpha=0.7, 
                   label=f'Breakeven = ${K+call_premium}')
axes[0, 1].fill_between(S_range, 0, call_profits, where=(call_profits >= 0), 
                        alpha=0.2, color='green', label='Profit Zone')
axes[0, 1].fill_between(S_range, 0, call_profits, where=(call_profits < 0), 
                        alpha=0.2, color='red', label='Loss Zone')
axes[0, 1].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[0, 1].set_ylabel('Profit/Loss ($)', fontsize=11)
axes[0, 1].set_title('Long Call Profit/Loss', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Put Payoff
axes[1, 0].plot(S_range, put_payoffs, linewidth=2.5, color='#F18F01', label='Put Payoff')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[1, 0].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[1, 0].fill_between(S_range, 0, put_payoffs, alpha=0.2, color='#F18F01')
axes[1, 0].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[1, 0].set_ylabel('Payoff ($)', fontsize=11)
axes[1, 0].set_title('Long Put Payoff', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Put Profit
axes[1, 1].plot(S_range, put_profits, linewidth=2.5, color='#6A994E', label='Put Profit')
axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)
axes[1, 1].axvline(x=K, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K}')
axes[1, 1].axvline(x=K-put_premium, color='green', linestyle='--', linewidth=1.5, alpha=0.7, 
                   label=f'Breakeven = ${K-put_premium}')
axes[1, 1].fill_between(S_range, 0, put_profits, where=(put_profits >= 0), 
                        alpha=0.2, color='green', label='Profit Zone')
axes[1, 1].fill_between(S_range, 0, put_profits, where=(put_profits < 0), 
                        alpha=0.2, color='red', label='Loss Zone')
axes[1, 1].set_xlabel('Underlying Price at Expiration ($)', fontsize=11)
axes[1, 1].set_ylabel('Profit/Loss ($)', fontsize=11)
axes[1, 1].set_title('Long Put Profit/Loss', fontsize=13, fontweight='bold')
axes[1, 1].legend(loc='upper right', fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("[OK] Payoff diagrams demonstrate the asymmetric risk/reward profiles of options")

#### Practical Example

Let's apply call and put options to a real-world scenario...

In [ ]:
# Practical example for Call and Put Options

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 3. Option Moneyness (ITM, ATM, OTM)

#### Theory

**Moneyness** describes the relationship between the current underlying price and the strike price, indicating whether an option has intrinsic value.

##### For Call Options:

- **In-the-Money (ITM)**: $S_0 > K$ - The option has intrinsic value; exercising would be profitable
- **At-the-Money (ATM)**: $S_0 \approx K$ - The option has minimal intrinsic value
- **Out-of-the-Money (OTM)**: $S_0 < K$ - The option has no intrinsic value; only time value

##### For Put Options:

- **In-the-Money (ITM)**: $S_0 < K$ - The option has intrinsic value
- **At-the-Money (ATM)**: $S_0 \approx K$ - The option has minimal intrinsic value  
- **Out-of-the-Money (OTM)**: $S_0 > K$ - The option has no intrinsic value

#### Trading Implications

**ITM Options:**
- Higher premium (more expensive)
- Higher delta (more sensitive to underlying price changes)
- More likely to be exercised
- Lower leverage but higher probability of profit

**ATM Options:**
- Maximum time value
- Delta around 0.50 for calls, -0.50 for puts
- Most actively traded (highest liquidity)
- Balance between cost and leverage

**OTM Options:**
- Lower premium (cheaper)
- Lower delta (less sensitive to price changes)
- Higher leverage potential
- Higher risk of expiring worthless

#### Practical Considerations

Market makers and traders often refer to moneyness using percentage terms:
- 10% ITM call: $S_0 = 1.10 \times K$
- 5% OTM put: $S_0 = 1.05 \times K$

## Python implementation for classifying option moneyness

def classify_moneyness(S0, K, option_type='call', atm_threshold=0.02):
    """
    Classify option moneyness (ITM, ATM, OTM).
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
    atm_threshold : float, default 0.02
        Percentage threshold for ATM classification (2%)
        
    Returns
    -------
    str
        Moneyness classification: 'ITM', 'ATM', or 'OTM'
        
    Examples
    --------
    >>> classify_moneyness(105, 100, 'call')
    'ITM'
    >>> classify_moneyness(95, 100, 'put')
    'ITM'
    """
    if option_type not in ['call', 'put']:
        raise ValueError("option_type must be 'call' or 'put'")
    
    if S0 <= 0 or K <= 0:
        raise ValueError("Prices must be positive")
    
    # Calculate moneyness ratio
    ratio = S0 / K
    
    # Check if ATM (within threshold)
    if abs(ratio - 1.0) <= atm_threshold:
        return 'ATM'
    
    # For call options
    if option_type == 'call':
        if ratio > 1.0:
            return 'ITM'
        else:
            return 'OTM'
    # For put options
    else:
        if ratio < 1.0:
            return 'ITM'
        else:
            return 'OTM'

def calculate_intrinsic_value(S0, K, option_type='call'):
    """
    Calculate intrinsic value of an option.
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
        
    Returns
    -------
    float
        Intrinsic value (non-negative)
    """
    if option_type == 'call':
        return max(S0 - K, 0)
    elif option_type == 'put':
        return max(K - S0, 0)
    else:
        raise ValueError("option_type must be 'call' or 'put'")

def calculate_moneyness_percentage(S0, K, option_type='call'):
    """
    Calculate moneyness as percentage.
    
    Parameters
    ----------
    S0 : float
        Current underlying price
    K : float
        Strike price
    option_type : str, default 'call'
        Option type: 'call' or 'put'
        
    Returns
    -------
    float
        Moneyness percentage (positive for ITM, negative for OTM)
        
    Examples
    --------
    >>> calculate_moneyness_percentage(110, 100, 'call')
    10.0  # 10% ITM
    """
    if option_type == 'call':
        return (S0 / K - 1) * 100
    else:  # put
        return (K / S0 - 1) * 100

## Example: Classify multiple options
print("=" * 70)
print("OPTION MONEYNESS CLASSIFICATION EXAMPLES")
print("=" * 70)

## Current underlying price
S0 = 100

## Various strike prices
strikes = [80, 90, 95, 98, 100, 102, 105, 110, 120]

print(f"\nCurrent Underlying Price: ${S0}\n")
print("-" * 70)
print(f"{'Strike':<10} {'Call':<15} {'Call IV':<12} {'Put':<15} {'Put IV':<12}")
print("-" * 70)

for K in strikes:
    call_class = classify_moneyness(S0, K, 'call')
    put_class = classify_moneyness(S0, K, 'put')
    call_iv = calculate_intrinsic_value(S0, K, 'call')
    put_iv = calculate_intrinsic_value(S0, K, 'put')
    
    print(f"${K:<9} {call_class:<15} ${call_iv:<11.2f} {put_class:<15} ${put_iv:<11.2f}")

print("-" * 70)

## Detailed example
print("\n" + "=" * 70)
print("DETAILED EXAMPLE: AAPL Options")
print("=" * 70)
S_AAPL = 150
K_AAPL = 145
call_premium_AAPL = 8.50
put_premium_AAPL = 2.30

print(f"Stock Price: ${S_AAPL}")
print(f"Strike Price: ${K_AAPL}")
print(f"Call Premium: ${call_premium_AAPL}")
print(f"Put Premium: ${put_premium_AAPL}")
print("-" * 70)

## Call analysis
call_moneyness = classify_moneyness(S_AAPL, K_AAPL, 'call')
call_intrinsic = calculate_intrinsic_value(S_AAPL, K_AAPL, 'call')
call_time_value = call_premium_AAPL - call_intrinsic
call_pct = calculate_moneyness_percentage(S_AAPL, K_AAPL, 'call')

print(f"\nCALL OPTION ANALYSIS:")
print(f"  Moneyness: {call_moneyness} ({call_pct:+.2f}%)")
print(f"  Intrinsic Value: ${call_intrinsic:.2f}")
print(f"  Time Value: ${call_time_value:.2f}")
print(f"  Time Value %: {(call_time_value/call_premium_AAPL)*100:.1f}% of premium")

## Put analysis
put_moneyness = classify_moneyness(S_AAPL, K_AAPL, 'put')
put_intrinsic = calculate_intrinsic_value(S_AAPL, K_AAPL, 'put')
put_time_value = put_premium_AAPL - put_intrinsic
put_pct = calculate_moneyness_percentage(S_AAPL, K_AAPL, 'put')

print(f"\nPUT OPTION ANALYSIS:")
print(f"  Moneyness: {put_moneyness} ({put_pct:+.2f}%)")
print(f"  Intrinsic Value: ${put_intrinsic:.2f}")
print(f"  Time Value: ${put_time_value:.2f}")
print(f"  Time Value %: {(put_time_value/put_premium_AAPL)*100:.1f}% of premium")

print("=" * 70)
print("\n[OK] Moneyness classification and intrinsic value calculation complete")

In [ ]:
# Visualization 1: Moneyness Across Strikes for Calls and Puts

# Parameters
S0 = 100  # Current underlying price
strikes = np.linspace(70, 130, 100)

# Calculate intrinsic values
call_intrinsic = np.array([calculate_intrinsic_value(S0, K, 'call') for K in strikes])
put_intrinsic = np.array([calculate_intrinsic_value(S0, K, 'put') for K in strikes])

# Create figure with subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Call Intrinsic Value vs Strike
axes[0, 0].plot(strikes, call_intrinsic, linewidth=2.5, color='#2E86AB', label='Call Intrinsic Value')
axes[0, 0].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

# Shade regions
axes[0, 0].fill_between(strikes, 0, call_intrinsic, where=(strikes < S0), 
                        alpha=0.3, color='green', label='ITM Region')
axes[0, 0].fill_between(strikes, 0, call_intrinsic, where=(strikes >= S0), 
                        alpha=0.2, color='gray', label='OTM Region')

# Annotations
axes[0, 0].annotate('ITM\n(Intrinsic Value)', xy=(85, 10), fontsize=11, 
                   ha='center', color='darkgreen', fontweight='bold')
axes[0, 0].annotate('OTM\n(No Intrinsic Value)', xy=(115, 2), fontsize=11, 
                   ha='center', color='gray', fontweight='bold')

axes[0, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 0].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[0, 0].set_title('Call Option: Intrinsic Value vs Strike', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Put Intrinsic Value vs Strike
axes[0, 1].plot(strikes, put_intrinsic, linewidth=2.5, color='#F18F01', label='Put Intrinsic Value')
axes[0, 1].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

# Shade regions
axes[0, 1].fill_between(strikes, 0, put_intrinsic, where=(strikes > S0), 
                        alpha=0.3, color='green', label='ITM Region')
axes[0, 1].fill_between(strikes, 0, put_intrinsic, where=(strikes <= S0), 
                        alpha=0.2, color='gray', label='OTM Region')

# Annotations
axes[0, 1].annotate('OTM\n(No Intrinsic Value)', xy=(85, 2), fontsize=11, 
                   ha='center', color='gray', fontweight='bold')
axes[0, 1].annotate('ITM\n(Intrinsic Value)', xy=(115, 10), fontsize=11, 
                   ha='center', color='darkgreen', fontweight='bold')

axes[0, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 1].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[0, 1].set_title('Put Option: Intrinsic Value vs Strike', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Moneyness Classification - Call Options
K_discrete = np.array([80, 85, 90, 95, 98, 100, 102, 105, 110, 115, 120])
call_classes = [classify_moneyness(S0, K, 'call') for K in K_discrete]
call_iv_discrete = [calculate_intrinsic_value(S0, K, 'call') for K in K_discrete]

# Color mapping
colors_call = ['green' if c == 'ITM' else 'orange' if c == 'ATM' else 'gray' for c in call_classes]

axes[1, 0].bar(K_discrete, call_iv_discrete, width=3, color=colors_call, alpha=0.7, edgecolor='black')
axes[1, 0].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 0].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[1, 0].set_title('Call Option Moneyness Classification', fontsize=13, fontweight='bold')

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', alpha=0.7, label='ITM'),
                   Patch(facecolor='orange', alpha=0.7, label='ATM'),
                   Patch(facecolor='gray', alpha=0.7, label='OTM')]
axes[1, 0].legend(handles=legend_elements, loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Plot 4: Moneyness Classification - Put Options
put_classes = [classify_moneyness(S0, K, 'put') for K in K_discrete]
put_iv_discrete = [calculate_intrinsic_value(S0, K, 'put') for K in K_discrete]

# Color mapping
colors_put = ['green' if c == 'ITM' else 'orange' if c == 'ATM' else 'gray' for c in put_classes]

axes[1, 1].bar(K_discrete, put_iv_discrete, width=3, color=colors_put, alpha=0.7, edgecolor='black')
axes[1, 1].axvline(x=S0, color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 1].set_ylabel('Intrinsic Value ($)', fontsize=11)
axes[1, 1].set_title('Put Option Moneyness Classification', fontsize=13, fontweight='bold')
axes[1, 1].legend(handles=legend_elements, loc='upper left', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("[OK] Moneyness visualizations demonstrate ITM/ATM/OTM regions clearly")

## 4. Intrinsic Value vs Time Value

### Theory

Every option premium consists of two components:

**Intrinsic Value** = The immediate exercise value of the option
- For calls: $\max(S_0 - K, 0)$
- For puts: $\max(K - S_0, 0)$
- Cannot be negative
- Represents "moneyness"

**Time Value** = Premium - Intrinsic Value
- Represents the optionality and uncertainty
- Always non-negative
- Decreases as expiration approaches (theta decay)
- Maximum for ATM options
- Influenced by volatility, time to expiration, and interest rates

### Key Relationships

$$
\text{Option Premium} = \text{Intrinsic Value} + \text{Time Value}
$$

**Time Value depends on:**
1. **Time to Expiration**: More time = more uncertainty = higher time value
2. **Volatility**: Higher volatility = more potential for price movement = higher time value
3. **Moneyness**: ATM options have maximum time value
4. **Interest Rates**: Higher rates increase call time value, decrease put time value

### Why Time Value Matters

- **For Buyers**: Time value is the "insurance premium" you pay for optionality
- **For Sellers**: Time value is income that decays in your favor
- **For Traders**: Understanding time decay (theta) is crucial for strategy selection

### Time Decay Characteristics

- **OTM Options**: Mostly time value, decay accelerates near expiration
- **ATM Options**: Pure time value, highest absolute theta
- **ITM Options**: Less time value as % of premium, more stable pricing


## Visualization: Time Value Decay and Option Value Components

## Simplified Black-Scholes for illustration (we'll cover this in detail later)
def simple_bs_call(S, K, T, r, sigma):
    """Simplified Black-Scholes call price for visualization"""
    if T <= 0:
        return max(S - K, 0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def simple_bs_put(S, K, T, r, sigma):
    """Simplified Black-Scholes put price for visualization"""
    if T <= 0:
        return max(K - S, 0)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

## Parameters
S0 = 100
K = 100
r = 0.05
sigma = 0.25

## Create subplots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

## Plot 1: Time Decay for ATM Call Option
T_range = np.linspace(0.001, 1, 100)
call_prices = [simple_bs_call(S0, K, T, r, sigma) for T in T_range]
intrinsic_values = [max(S0 - K, 0) for _ in T_range]
time_values = [cp - iv for cp, iv in zip(call_prices, intrinsic_values)]

axes[0, 0].plot(T_range*252, call_prices, linewidth=2.5, color='#2E86AB', label='Total Premium')
axes[0, 0].plot(T_range*252, intrinsic_values, linewidth=2, color='green', linestyle='--', label='Intrinsic Value')
axes[0, 0].fill_between(T_range*252, intrinsic_values, call_prices, alpha=0.3, color='orange', label='Time Value')
axes[0, 0].set_xlabel('Days to Expiration', fontsize=11)
axes[0, 0].set_ylabel('Option Value ($)', fontsize=11)
axes[0, 0].set_title('ATM Call: Time Decay (S=$100, K=$100)', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper left', fontsize=10)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].invert_xaxis()  # Show time decay from left to right

## Plot 2: Value Components Across Strikes
strikes_comp = np.linspace(80, 120, 50)
T_fixed = 0.25  # 3 months

call_premiums = [simple_bs_call(S0, K, T_fixed, r, sigma) for K in strikes_comp]
call_intrinsics = [max(S0 - K, 0) for K in strikes_comp]
call_time_vals = [cp - ci for cp, ci in zip(call_premiums, call_intrinsics)]

axes[0, 1].fill_between(strikes_comp, 0, call_intrinsics, alpha=0.5, color='green', label='Intrinsic Value')
axes[0, 1].fill_between(strikes_comp, call_intrinsics, call_premiums, alpha=0.5, color='orange', label='Time Value')
axes[0, 1].plot(strikes_comp, call_premiums, linewidth=2.5, color='black', label='Total Premium')
axes[0, 1].axvline(x=S0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Spot = ${S0}')
axes[0, 1].set_xlabel('Strike Price ($)', fontsize=11)
axes[0, 1].set_ylabel('Value ($)', fontsize=11)
axes[0, 1].set_title('Call Option: Intrinsic vs Time Value (T=3mo)', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper right', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

## Plot 3: Time Value as Percentage of Premium
time_val_pct = [(tv/cp)*100 if cp > 0 else 0 for tv, cp in zip(call_time_vals, call_premiums)]

axes[1, 0].plot(strikes_comp, time_val_pct, linewidth=2.5, color='#A23B72')
axes[1, 0].axvline(x=S0, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Spot = ${S0}')
axes[1, 0].axhline(y=100, color='gray', linestyle=':', linewidth=1, alpha=0.7, label='100% Time Value')
axes[1, 0].fill_between(strikes_comp, 0, time_val_pct, alpha=0.3, color='#A23B72')

## Annotate key regions
axes[1, 0].annotate('ATM:\n100% Time Value', xy=(100, 100), xytext=(105, 90),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                   fontsize=10, fontweight='bold')
axes[1, 0].annotate('Deep ITM:\nMostly Intrinsic', xy=(85, 20), xytext=(85, 40),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                   fontsize=10, fontweight='bold')

axes[1, 0].set_xlabel('Strike Price ($)', fontsize=11)
axes[1, 0].set_ylabel('Time Value (% of Premium)', fontsize=11)
axes[1, 0].set_title('Time Value as Percentage of Total Premium', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 110])

## Plot 4: Time Decay Acceleration
T_decay = np.linspace(0.001, 1, 200)
strikes_decay = [90, 100, 110]  # OTM, ATM, ITM

for K_decay in strikes_decay:
    prices = [simple_bs_call(S0, K_decay, T, r, sigma) for T in T_decay]
    intrinsics = [max(S0 - K_decay, 0) for _ in T_decay]
    time_vals = [p - i for p, i in zip(prices, intrinsics)]
    
    moneyness_label = 'ATM' if K_decay == 100 else ('OTM' if K_decay > 100 else 'ITM')
    axes[1, 1].plot(T_decay*252, time_vals, linewidth=2.5, label=f'K=${K_decay} ({moneyness_label})')

axes[1, 1].set_xlabel('Days to Expiration', fontsize=11)
axes[1, 1].set_ylabel('Time Value ($)', fontsize=11)
axes[1, 1].set_title('Time Decay Across Different Strikes (S=$100)', fontsize=13, fontweight='bold')
axes[1, 1].legend(loc='upper left', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].invert_xaxis()

## Shade final 30 days to show acceleration
axes[1, 1].axvspan(30, 0, alpha=0.2, color='red', label='Accelerated Decay')
axes[1, 1].text(15, axes[1, 1].get_ylim()[1]*0.9, 'Decay\nAccelerates', 
               ha='center', fontsize=10, fontweight='bold', color='darkred')

plt.tight_layout()
plt.show()

print("[OK] Time value and decay visualizations show premium components clearly")
print("KEY INSIGHT: ATM options have maximum time value, which decays fastest near expiration")

In [ ]:
# Practical example for Strike Price, Expiration, Premium

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 4. Moneyness (ITM, ATM, OTM)

### Theory

Detailed explanation of moneyness (itm, atm, otm)...

#### Mathematical Formulation

The mathematical framework for moneyness (itm, atm, otm) is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Moneyness (ITM, ATM, OTM)

def compute_moneyness_itm_atm_otm():
    """
    Moneyness (ITM, ATM, OTM) implementation
    """
    # Implementation code here
    pass

print(f'[OK] Moneyness (ITM, ATM, OTM) implementation complete')

In [ ]:
# Visualization for Moneyness (ITM, ATM, OTM)

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Moneyness (ITM, ATM, OTM)')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply moneyness (itm, atm, otm) to a real-world scenario...

In [ ]:
# Practical example for Moneyness (ITM, ATM, OTM)

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 5. Payoff Diagrams

### Theory

Detailed explanation of payoff diagrams...

#### Mathematical Formulation

The mathematical framework for payoff diagrams is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Payoff Diagrams

def compute_payoff_diagrams():
    """
    Payoff Diagrams implementation
    """
    # Implementation code here
    pass

print(f'[OK] Payoff Diagrams implementation complete')

In [ ]:
# Visualization for Payoff Diagrams

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Payoff Diagrams')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply payoff diagrams to a real-world scenario...

In [ ]:
# Practical example for Payoff Diagrams

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 5. European vs American Options

#### Theory

Options are classified by their **exercise style** - when the holder can exercise the option:

##### European Options

- **Can ONLY be exercised at expiration**
- Simpler to price (closed-form solutions like Black-Scholes)
- Common in index options (SPX, VIX)
- Most OTC (over-the-counter) options
- No early exercise premium in price

**Mathematical Note**: For European options:
$$
V_{\text{European}}(t) = e^{-r(T-t)} \mathbb{E}^{\mathbb{Q}}[\text{Payoff}(S_T) | \mathcal{F}_t]
$$

##### American Options

- **Can be exercised at ANY time up to and including expiration**
- More valuable than European (never less) due to early exercise optionality
- Common in equity options (most US exchange-traded options)
- Require numerical methods for pricing (binomial trees, finite difference)
- Early exercise is optimal in specific situations

**Mathematical Note**: For American options:
$$
V_{\text{American}}(t) = \max\left(\text{Immediate Exercise}, \text{Continuation Value}\right)
$$

#### Early Exercise Considerations

**For American Call Options on Non-Dividend Paying Stocks:**
- Early exercise is typically **NOT optimal**
- Reason: Time value > intrinsic value before expiration
- Exception: Deep ITM calls approaching expiration with very low time value

**For American Put Options:**
- Early exercise **CAN be optimal**
- Reason: Put payoff is bounded (max = K), time value of money matters
- Optimal when: Deep ITM and interest earned on K exceeds time value lost

**For Dividend-Paying Stocks:**
- American calls may be exercised early **just before ex-dividend date**
- Reason: Capture dividend vs holding option (stock price drops by dividend)
- Trade-off: Dividend captured vs time value sacrificed

#### Key Differences Summary

| Feature | European | American |
|---------|----------|----------|
| **Exercise** | Only at expiration | Anytime until expiration |
| **Value** | Lower | Higher (≥ European) |
| **Pricing** | Closed-form (B-S) | Numerical methods |
| **Common Use** | Index options, OTC | Equity options |
| **Early Exercise** | Not applicable | Sometimes optimal |

#### Practical Implications

1. **For Option Buyers**: American options provide more flexibility but may cost more
2. **For Option Sellers**: Need to be aware of assignment risk at any time
3. **For Pricing**: American options require more sophisticated models
4. **For Risk Management**: Early exercise creates additional considerations

## Visualization: European vs American Options and Early Exercise

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

## Plot 1: American vs European Put Value
S_range = np.linspace(50, 150, 100)
K_put = 100
T_put = 0.5
r_put = 0.05
sigma_put = 0.30

## European put values
euro_put_values = [simple_bs_put(S, K_put, T_put, r_put, sigma_put) for S in S_range]

## Intrinsic values (lower bound for American)
intrinsic_put = [max(K_put - S, 0) for S in S_range]

## American put (approximation: max of European and intrinsic)
american_put_values = [max(ev, iv) for ev, iv in zip(euro_put_values, intrinsic_put)]

axes[0, 0].plot(S_range, american_put_values, linewidth=2.5, color='#2E86AB', label='American Put')
axes[0, 0].plot(S_range, euro_put_values, linewidth=2.5, color='#F18F01', linestyle='--', label='European Put')
axes[0, 0].plot(S_range, intrinsic_put, linewidth=2, color='green', linestyle=':', label='Intrinsic Value (Exercise)')
axes[0, 0].fill_between(S_range, euro_put_values, american_put_values, alpha=0.3, color='purple', label='Early Exercise Premium')

axes[0, 0].axvline(x=K_put, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_put}')
axes[0, 0].set_xlabel('Stock Price ($)', fontsize=11)
axes[0, 0].set_ylabel('Option Value ($)', fontsize=11)
axes[0, 0].set_title('American vs European Put Option (K=$100, T=6mo)', fontsize=13, fontweight='bold')
axes[0, 0].legend(loc='upper right', fontsize=9)
axes[0, 0].grid(True, alpha=0.3)

## Annotate early exercise region
axes[0, 0].annotate('Early Exercise\nOptimal Here', xy=(65, 32), xytext=(70, 40),
                   arrowprops=dict(arrowstyle='->', color='purple', lw=2),
                   fontsize=10, fontweight='bold', color='purple')

## Plot 2: Early Exercise Boundary for American Put
T_boundary = np.linspace(0.01, 1, 100)
## Simplified early exercise boundary (actual calculation is complex)
S_critical = K_put * (1 - 0.15 * np.sqrt(T_boundary))  # Approximation

axes[0, 1].fill_between(T_boundary*252, 0, S_critical, alpha=0.3, color='red', label='Early Exercise Region')
axes[0, 1].fill_between(T_boundary*252, S_critical, 150, alpha=0.3, color='green', label='Hold Option Region')
axes[0, 1].plot(T_boundary*252, S_critical, linewidth=3, color='black', label='Exercise Boundary')
axes[0, 1].axhline(y=K_put, color='blue', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_put}')

axes[0, 1].set_xlabel('Days to Expiration', fontsize=11)
axes[0, 1].set_ylabel('Stock Price ($)', fontsize=11)
axes[0, 1].set_title('American Put: Early Exercise Boundary', fontsize=13, fontweight='bold')
axes[0, 1].legend(loc='upper left', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim([0, 150])
axes[0, 1].invert_xaxis()

axes[0, 1].text(125, 70, 'Exercise if stock\nprice falls below\nboundary', 
               ha='center', fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

## Plot 3: Dividend Impact on Call Early Exercise
S_div = 100
K_call = 95
T_div = 0.25  # 3 months
dividend = 3  # $3 dividend
ex_div_days = 60  # 60 days from now

## Timeline
days = np.array([0, ex_div_days-1, ex_div_days, 90])
stock_prices = np.array([S_div, S_div, S_div-dividend, S_div-dividend])
call_values_no_exercise = np.array([8, 10, 7.5, 8.5])  # Hypothetical
intrinsic_before_div = S_div - K_call
exercise_value = intrinsic_before_div  # Value if exercised just before dividend

axes[1, 0].plot(days, stock_prices, linewidth=2.5, color='#2E86AB', marker='o', markersize=8, label='Stock Price')
axes[1, 0].axvline(x=ex_div_days, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Ex-Dividend Date')
axes[1, 0].axhline(y=K_call, color='green', linestyle=':', linewidth=1.5, alpha=0.7, label=f'Strike = ${K_call}')

## Annotate dividend drop
axes[1, 0].annotate(f'Dividend Drop\n${dividend}', xy=(ex_div_days, S_div-dividend/2), xytext=(ex_div_days+10, S_div+2),
                   arrowprops=dict(arrowstyle='->', color='red', lw=2),
                   fontsize=10, fontweight='bold', color='red')

axes[1, 0].set_xlabel('Days from Today', fontsize=11)
axes[1, 0].set_ylabel('Stock Price ($)', fontsize=11)
axes[1, 0].set_title('Stock Price Around Ex-Dividend Date', fontsize=13, fontweight='bold')
axes[1, 0].legend(loc='upper right', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

## Plot 4: Early Exercise Decision Tree
axes[1, 1].axis('off')

decision_text = """
EARLY EXERCISE DECISION GUIDE

American CALL (Non-Dividend Stock):
  ├─ Almost never optimal
  ├─ Time value > 0 before expiration
  └─ Better to sell than exercise

American CALL (Dividend-Paying Stock):
  ├─ Consider before ex-dividend date
  ├─ Exercise if: Dividend > Time Value
  └─ Trade-off: Capture div vs lose optionality

American PUT:
  ├─ May be optimal when deep ITM
  ├─ Exercise if: Interest on K > Time Value
  ├─ More likely when:
  │   ├─ High interest rates
  │   ├─ Low volatility
  │   └─ Deep in-the-money
  └─ Time value approaches zero

European Options:
  └─ No early exercise allowed
      └─ Hold until expiration

PRACTICAL RULE OF THUMB:
  • Calls: Rarely exercise early (unless dividend)
  • Puts: Sometimes optimal when deep ITM
  • Check: Intrinsic Value vs Option Market Price
  • If IV > Market Price → Consider exercise
  • If Market Price > IV → Sell instead
"""

axes[1, 1].text(0.05, 0.95, decision_text, transform=axes[1, 1].transAxes,
               fontsize=9.5, verticalalignment='top', family='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.show()

print("[OK] European vs American visualizations complete")
print("KEY INSIGHT: American options have early exercise flexibility, most valuable for puts and dividend-paying calls")

In [ ]:
# Python implementation for European vs American Options

def compute_european_vs_american_options():
    """
    European vs American Options implementation
    """
    # Implementation code here
    pass

print(f'[OK] European vs American Options implementation complete')

In [ ]:
# Visualization for European vs American Options

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('European vs American Options')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply european vs american options to a real-world scenario...

In [ ]:
# Practical example for European vs American Options

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 7. Exercise and Assignment

### Theory

Detailed explanation of exercise and assignment...

#### Mathematical Formulation

The mathematical framework for exercise and assignment is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Exercise and Assignment

def compute_exercise_and_assignment():
    """
    Exercise and Assignment implementation
    """
    # Implementation code here
    pass

print(f'[OK] Exercise and Assignment implementation complete')

In [ ]:
# Visualization for Exercise and Assignment

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Exercise and Assignment')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply exercise and assignment to a real-world scenario...

In [ ]:
# Practical example for Exercise and Assignment

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### Comprehensive Case Study: Trading Options on TechStock Inc.

#### Scenario

**Company**: TechStock Inc. (Ticker: TECH)  
**Current Stock Price**: $125.00  
**Date**: September 1, 2024  
**Volatility**: 30% annualized  
**Risk-free Rate**: 5% annualized

You are analyzing the following October 2024 options (45 days to expiration):

| Strike | Call Premium | Put Premium |
|--------|--------------|-------------|
| $115   | $12.50       | $1.80       |
| $120   | $8.75        | $2.90       |
| $125   | $5.50        | $4.75       |
| $130   | $3.20        | $7.40       |
| $135   | $1.75        | $11.15      |

#### Analysis Tasks

Let's perform a comprehensive analysis of these options using all concepts covered.

In [ ]:
# Comprehensive Case Study: Analysis and Trading Strategies

# Market data
S0_case = 125.00
strikes_case = np.array([115, 120, 125, 130, 135])
call_premiums = np.array([12.50, 8.75, 5.50, 3.20, 1.75])
put_premiums = np.array([1.80, 2.90, 4.75, 7.40, 11.15])
T_case = 45/252  # 45 days in years
sigma_case = 0.30
r_case = 0.05

print("=" * 80)
print("COMPREHENSIVE OPTIONS ANALYSIS: TechStock Inc.")
print("=" * 80)
print(f"Stock Price: ${S0_case:.2f}")
print(f"Days to Expiration: 45")
print(f"Volatility: {sigma_case*100:.0f}%")
print(f"Risk-free Rate: {r_case*100:.0f}%")
print("=" * 80)

# Task 1: Classify Moneyness
print("\n1. MONEYNESS CLASSIFICATION")
print("-" * 80)
print(f"{'Strike':<10} {'Call':<20} {'Put':<20} {'Call IV':<15} {'Put IV':<15}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    call_class = classify_moneyness(S0_case, K, 'call')
    put_class = classify_moneyness(S0_case, K, 'put')
    call_iv = calculate_intrinsic_value(S0_case, K, 'call')
    put_iv = calculate_intrinsic_value(S0_case, K, 'put')
    
    print(f"${K:<9} {call_class:<20} {put_class:<20} ${call_iv:<14.2f} ${put_iv:<14.2f}")

# Task 2: Decompose Premiums
print("\n2. PREMIUM DECOMPOSITION (Intrinsic vs Time Value)")
print("-" * 80)
print(f"{'Strike':<10} {'Option':<8} {'Premium':<12} {'Intrinsic':<12} {'Time Value':<12} {'TV %':<10}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    # Call analysis
    call_iv = calculate_intrinsic_value(S0_case, K, 'call')
    call_tv = call_premiums[i] - call_iv
    call_tv_pct = (call_tv / call_premiums[i]) * 100
    print(f"${K:<9} {'Call':<8} ${call_premiums[i]:<11.2f} ${call_iv:<11.2f} ${call_tv:<11.2f} {call_tv_pct:<9.1f}%")
    
    # Put analysis
    put_iv = calculate_intrinsic_value(S0_case, K, 'put')
    put_tv = put_premiums[i] - put_iv
    put_tv_pct = (put_tv / put_premiums[i]) * 100
    print(f"${K:<9} {'Put':<8} ${put_premiums[i]:<11.2f} ${put_iv:<11.2f} ${put_tv:<11.2f} {put_tv_pct:<9.1f}%")
    print()

# Task 3: Calculate Payoffs and Profits at Various Stock Prices
print("\n3. PAYOFF ANALYSIS AT EXPIRATION")
print("-" * 80)

expiration_prices = np.array([110, 115, 120, 125, 130, 135, 140])
print(f"\n{'Stock':<10}", end="")
for K in strikes_case:
    print(f"K=${K} Call  ", end="")
print()
print("-" * 80)

for S_T in expiration_prices:
    print(f"${S_T:<9}", end="")
    for K in strikes_case:
        profit = call_profit(S_T, K, call_premiums[strikes_case == K][0])
        print(f"${profit:>7.2f}{'':>4}", end="")
    print()

# Task 4: Identify Best Strategy for Different Market Views
print("\n\n4. TRADING STRATEGY RECOMMENDATIONS")
print("-" * 80)

strategies = {
    'BULLISH (Expect price to rise to $135+)': {
        'strategy': 'Buy $130 Call',
        'cost': call_premiums[strikes_case == 130][0],
        'breakeven': 130 + call_premiums[strikes_case == 130][0],
        'max_loss': call_premiums[strikes_case == 130][0],
        'max_gain': 'Unlimited',
        'rationale': 'OTM call provides high leverage with defined risk'
    },
    'MODERATELY BULLISH (Expect price to $130)': {
        'strategy': 'Buy $125 Call',
        'cost': call_premiums[strikes_case == 125][0],
        'breakeven': 125 + call_premiums[strikes_case == 125][0],
        'max_loss': call_premiums[strikes_case == 125][0],
        'max_gain': 'Unlimited',
        'rationale': 'ATM call balances cost and probability'
    },
    'BEARISH (Expect price to fall to $115-)': {
        'strategy': 'Buy $125 Put',
        'cost': put_premiums[strikes_case == 125][0],
        'breakeven': 125 - put_premiums[strikes_case == 125][0],
        'max_loss': put_premiums[strikes_case == 125][0],
        'max_gain': f'${125 - put_premiums[strikes_case == 125][0]:.2f}',
        'rationale': 'ATM put provides downside protection'
    },
    'NEUTRAL (Expect price to stay around $125)': {
        'strategy': 'Sell $120 Put + Sell $130 Call',
        'cost': -(put_premiums[strikes_case == 120][0] + call_premiums[strikes_case == 130][0]),
        'breakeven': '120 or 130',
        'max_loss': 'Significant if price moves',
        'max_gain': put_premiums[strikes_case == 120][0] + call_premiums[strikes_case == 130][0],
        'rationale': 'Short strangle collects premium if price stays range-bound'
    }
}

for view, details in strategies.items():
    print(f"\n{view}")
    print(f"  Strategy: {details['strategy']}")
    print(f"  Cost/Credit: ${abs(details['cost']):.2f} {'debit' if details['cost'] > 0 else 'credit'}")
    print(f"  Breakeven: ${details['breakeven']}")
    print(f"  Max Loss: ${details['max_loss']}")
    print(f"  Max Gain: {details['max_gain']}")
    print(f"  Rationale: {details['rationale']}")

# Task 5: Calculate Required Move for Profitability
print("\n\n5. REQUIRED PRICE MOVES FOR PROFITABILITY")
print("-" * 80)
print(f"{'Strike':<10} {'Call BE':<15} {'% Move':<15} {'Put BE':<15} {'% Move':<15}")
print("-" * 80)

for i, K in enumerate(strikes_case):
    call_be = K + call_premiums[i]
    call_move_pct = ((call_be - S0_case) / S0_case) * 100
    
    put_be = K - put_premiums[i]
    put_move_pct = ((put_be - S0_case) / S0_case) * 100
    
    print(f"${K:<9} ${call_be:<14.2f} {call_move_pct:>6.2f}%{'':>7} ${put_be:<14.2f} {put_move_pct:>6.2f}%")

print("\n" + "=" * 80)
print("CASE STUDY COMPLETE")
print("=" * 80)
print("\nKEY TAKEAWAYS:")
print("1. ATM options have maximum time value (~100% of premium)")
print("2. ITM options have more intrinsic value, less time value")
print("3. OTM options are cheaper but require larger moves to profit")
print("4. Option selection depends on market view, risk tolerance, and timeframe")
print("5. Always consider breakeven points before entering trades")
print("=" * 80)

### Practice Exercises

Test your understanding with these exercises:

### Exercise 1\nDescription of exercise 1...

### Exercise 2\nDescription of exercise 2...

### Exercise 3\nDescription of exercise 3...

In [ ]:
# Your solution for Exercise 1



### Summary and Key Takeaways

#### Fundamental Concepts Mastered

**1. Call and Put Options**
- **Calls** give the right to BUY at the strike price
- **Puts** give the right to SELL at the strike price
- Options provide asymmetric risk/reward profiles
- Payoffs: $\max(S_T - K, 0)$ for calls, $\max(K - S_T, 0)$ for puts
- Limited loss for buyers, potentially unlimited for sellers

**2. Option Moneyness (ITM, ATM, OTM)**
- Moneyness describes the relationship between spot and strike
- **ITM**: Has intrinsic value, higher premium, lower leverage
- **ATM**: Maximum time value, most liquid, ~50% probability
- **OTM**: Cheaper, higher leverage, greater risk of expiring worthless
- Classification differs for calls (S > K is ITM) vs puts (S < K is ITM)

**3. Intrinsic Value vs Time Value**
- Premium = Intrinsic Value + Time Value
- Intrinsic value = immediate exercise value (never negative)
- Time value = optionality, uncertainty, and waiting benefit
- Time value maximized for ATM options
- Time decay (theta) accelerates near expiration

**4. European vs American Options**
- European: Exercise only at expiration
- American: Exercise anytime until expiration
- American options ≥ European options in value
- Early exercise rarely optimal for calls (except dividends)
- Early exercise can be optimal for puts (deep ITM)

**5. Exercise and Assignment Mechanics**
- Exercise is voluntary for option buyers
- Assignment is involuntary for option sellers
- Options automatically exercised if ITM at expiration (typically $0.01+)
- Consider transaction costs and taxes when deciding to exercise
- Usually better to sell than exercise before expiration (capture time value)

**6. Practical Trading Considerations**
- Select strikes based on market view and risk tolerance
- ATM options offer best balance of cost and probability
- OTM options provide leverage but lower probability
- Time decay works against buyers, in favor of sellers
- Always calculate breakeven prices before entering trades

#### Critical Formulas

$$
\begin{align*}
\text{Call Payoff} &= \max(S_T - K, 0) \\
\text{Put Payoff} &= \max(K - S_T, 0) \\
\text{Premium} &= \text{Intrinsic Value} + \text{Time Value} \\
\text{Breakeven (Call)} &= K + \text{Premium} \\
\text{Breakeven (Put)} &= K - \text{Premium} \\
\text{Moneyness Ratio} &= S_0 / K
\end{align*}
$$

#### Next Steps for Learning

1. **Option Greeks**: Learn how options prices change with underlying price (delta), volatility (vega), time (theta), and rates (rho)
2. **Option Pricing Models**: Black-Scholes, binomial trees, Monte Carlo simulation
3. **Implied Volatility**: How to extract market expectations from option prices
4. **Option Strategies**: Spreads, straddles, strangles, butterflies, condors
5. **Risk Management**: Delta hedging, gamma risk, position sizing
6. **Volatility Surface**: Understanding the smile and skew

#### Further Reading

**Academic Textbooks:**
- Hull, J.C. (2022). *Options, Futures, and Other Derivatives*, 11th Edition. Pearson Education.
- McDonald, R.L. (2013). *Derivatives Markets*, 3rd Edition. Pearson.
- Shreve, S.E. (2004). *Stochastic Calculus for Finance II: Continuous-Time Models*. Springer.

**Practitioner Books:**
- Natenberg, S. (2015). *Option Volatility and Pricing*, 2nd Edition. McGraw-Hill.
- McMillan, L.G. (2012). *Options as a Strategic Investment*, 5th Edition. Prentice Hall.

**Online Resources:**
- CBOE Options Institute: [www.cboe.com/education](https://www.cboe.com/education)
- Options Industry Council (OIC): [www.optionseducation.org](https://www.optionseducation.org)
- QuantLib Documentation: [www.quantlib.org](https://www.quantlib.org)

**Research Papers:**
- Black, F., & Scholes, M. (1973). "The Pricing of Options and Corporate Liabilities". *Journal of Political Economy*, 81(3), 637-654.
- Merton, R.C. (1973). "Theory of Rational Option Pricing". *Bell Journal of Economics and Management Science*, 4(1), 141-183.

#### Practice Recommendations

1. **Paper Trading**: Practice with virtual money before risking capital
2. **Track Real Options**: Monitor how premiums change with underlying price and time
3. **Build Models**: Implement option pricing and payoff calculators
4. **Study Market Data**: Analyze option chains and implied volatility patterns
5. **Join Communities**: Engage with options traders on forums and social media

---

**Congratulations!** You've completed the Options Basics notebook. You now have a solid foundation in options terminology, mechanics, and basic strategies. This knowledge forms the bedrock for more advanced topics in derivatives pricing, trading, and risk management.

**Estimated Completion Time**: 90-120 minutes

---## References### Academic Papers and Books1. Hull, J.C. (2021). *Options, Futures, and Other Derivatives (11th ed.)*. Pearson. Comprehensive derivatives textbook.2. Wilmott, P. (2006). *Paul Wilmott on Quantitative Finance*. Wiley. Three-volume set covering mathematical finance.3. Gatheral, J. (2006). *The Volatility Surface: A Practitioner's Guide*. Wiley. Essential volatility modeling reference.4. Andersen, T.G. et al. (2003). *Modeling and Forecasting Realized Volatility*. Econometrica, 71(2), 579-625.### Online Resources1. ***QuantLib Documentation*** - https://www.quantlib.org/ - Open-source quantitative finance library.2. ***Quantitative Finance on arXiv*** - https://arxiv.org/archive/q-fin - Latest research papers.3. ***Financial Python*** - https://github.com/quantopian - Quantopian lectures and tutorials.### Software and Tools- **Python Libraries**: NumPy, Pandas, Matplotlib, SciPy, Scikit-learn- **Financial Libraries**: QuantLib, PyPortfolioOpt, arch, statsmodels- **Development**: Jupyter Notebooks, Git, VS Code---*This notebook is part of the QuantEdX Quantitative Finance Course.**For questions or feedback, please contact: content@quantedx.com*

---

## Chapter 2: Option Trading Strategies

**Level:** Beginner\n**Concepts Covered:** 8

---

## Overview

This comprehensive notebook covers option trading strategies with detailed explanations, mathematical derivations, Python implementations, and practical examples.

### 1. Introduction

#### Why Option Trading Strategies Matter

Option trading strategies are essential tools for modern traders and portfolio managers. While individual options provide exposure to directional moves or volatility changes, **combinations of options** create sophisticated payoff structures that can:

- **Generate income** through premium collection while holding stock positions
- **Hedge portfolio risk** by providing downside protection at known costs
- **Express market views** with defined risk/reward profiles
- **Capitalize on volatility** expectations without directional bias
- **Reduce capital requirements** compared to outright stock positions

#### Types of Option Strategies

We categorize strategies into four main groups:

1. **Directional Strategies**: Bull/bear spreads that profit from price movements with limited risk
2. **Income Strategies**: Covered calls, iron condors that collect premium from time decay
3. **Volatility Strategies**: Straddles, strangles that profit from large moves in either direction
4. **Hedging Strategies**: Protective puts, collars that provide portfolio insurance

#### Risk Management Principles

Successful option trading requires disciplined risk management:

- **Position sizing**: Never risk more than 2-5% of portfolio on a single strategy
- **Maximum loss definition**: Every strategy should have clearly defined maximum loss
- **Breakeven analysis**: Understand the price levels where strategies become profitable
- **Greeks monitoring**: Track delta, gamma, theta, vega exposure continuously
- **Volatility awareness**: Implied volatility significantly impacts strategy profitability

#### Learning Objectives

By the end of this notebook, you will be able to:

1. Construct and analyze payoff diagrams for 8 core option strategies
2. Calculate maximum profit, maximum loss, and breakeven points
3. Implement strategies in Python with realistic market parameters
4. Compute strategy-level Greeks for risk management
5. Select appropriate strategies based on market outlook and risk tolerance
6. Apply strategies in a comprehensive portfolio management case study

#### Strategies Covered

1. **Covered Call** - Income generation from long stock position
2. **Protective Put** - Portfolio insurance and downside protection
3. **Bull Call Spread** - Bullish position with defined risk/reward
4. **Bear Put Spread** - Bearish position with limited capital outlay
5. **Straddle/Strangle** - Volatility plays for large directional moves
6. **Iron Condor** - Market-neutral income strategy
7. **Butterfly Spread** - Precision bet on price stability
8. **Calendar Spread** - Time decay arbitrage between maturities

**Prerequisites**: Understanding of option basics, payoff structures, and Black-Scholes pricing (covered in notebook 01_options_basics)

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Visualization settings - using colorblind-friendly palette
plt.style.use('seaborn-v0_8-whitegrid')
colors = ['#0173B2', '#DE8F05', '#029E73', '#CC78BC', '#CA9161', '#949494', '#ECE133', '#56B4E9']
sns.set_palette(colors)
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['legend.fontsize'] = 9
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

print('=' * 60)
print('Option Trading Strategies Notebook'.center(60))
print('=' * 60)
print('[OK] Libraries imported successfully')
print('[OK] Visualization settings configured')
print('[OK] Ready for strategy analysis')

### 2. Covered Call Strategy

#### Theory and Market Context

The **covered call** is one of the most popular option strategies, combining a long stock position with a short (sold) call option. This strategy is employed by investors who:

- **Own the underlying stock** and want to generate additional income
- Have a **neutral to slightly bullish** outlook on the stock
- Are willing to **cap their upside** in exchange for premium income
- Want to **reduce cost basis** of their stock position over time

**Key Characteristics:**
- **Maximum Profit**: Limited to strike price + premium received
- **Maximum Loss**: Stock can go to zero (minus premium received)
- **Breakeven**: Stock purchase price - premium received
- **Market View**: Neutral to slightly bullish
- **Volatility Impact**: Benefits from decreasing volatility (short option position)

**When to Use:**
1. You own stock and expect sideways to modest upward movement
2. You want to generate income from stocks in your portfolio
3. You're willing to sell your stock if it rises above the strike price
4. Implied volatility is elevated, making call premiums attractive

**Risks:**
- Limited upside if stock rallies significantly above strike
- Still exposed to downside risk (though premium provides cushion)
- Assignment risk if stock rises above strike before expiration
- Opportunity cost if stock surges and you're forced to sell

#### Mathematical Formulation

The covered call payoff at expiration is the combination of long stock and short call positions:

**Position Components:**
$$
\begin{align}
\text{Long Stock Position:} \quad & \Pi_{\text{stock}}(S_T) = S_T - S_0 \\
\text{Short Call Position:} \quad & \Pi_{\text{call}}(S_T) = -\max(S_T - K, 0) + C_0
\end{align}
$$

**Total Payoff:**
$$
\Pi_{\text{covered call}}(S_T) = (S_T - S_0) - \max(S_T - K, 0) + C_0
$$

**Simplified Form:**
$$
\Pi_{\text{covered call}}(S_T) = \begin{cases}
S_T - S_0 + C_0 & \text{if } S_T < K \\
K - S_0 + C_0 & \text{if } S_T \geq K
\end{cases}
$$

**Key Metrics:**

Maximum Profit:
$$
\Pi_{\max} = K - S_0 + C_0
$$

Maximum Loss:
$$
\Pi_{\min} = -S_0 + C_0
$$

Breakeven Point:
$$
S_T^* = S_0 - C_0
$$

**Greeks for Covered Call:**

Delta (price sensitivity):
$$
\Delta_{\text{covered call}} = 1 - \Delta_{\text{call}} \approx 1 - N(d_1)
$$

Theta (time decay):
$$
\Theta_{\text{covered call}} = -\Theta_{\text{call}} > 0
$$
The covered call has positive theta, benefiting from time decay.

Vega (volatility sensitivity):
$$
\nu_{\text{covered call}} = -\nu_{\text{call}} < 0
$$
The position benefits from decreasing implied volatility.

where:
- $S_0$ = initial stock price
- $S_T$ = stock price at expiration
- $K$ = call strike price
- $C_0$ = call premium received
- $N(d_1)$ = cumulative normal distribution of $d_1$ from Black-Scholes

In [ ]:
# Python implementation for Covered Call

def option_payoff(S, K, option_type='call', position='long'):
    """
    Calculate option payoff at expiration.
    
    Parameters
    ----------
    S : np.ndarray
        Array of stock prices at expiration
    K : float
        Strike price
    option_type : str, default='call'
        Type of option: 'call' or 'put'
    position : str, default='long'
        Position type: 'long' or 'short'
    
    Returns
    -------
    np.ndarray
        Payoff values for each stock price
    
    Examples
    --------
    >>> S = np.array([90, 100, 110])
    >>> payoff = option_payoff(S, K=100, option_type='call', position='long')
    >>> payoff
    array([0, 0, 10])
    """
    if option_type == 'call':
        intrinsic = np.maximum(S - K, 0)
    else:  # put
        intrinsic = np.maximum(K - S, 0)
    
    return intrinsic if position == 'long' else -intrinsic


def black_scholes_greeks(S, K, T, r, sigma, option_type='call'):
    """
    Calculate Black-Scholes Greeks for an option.
    
    Parameters
    ----------
    S : float
        Current stock price
    K : float
        Strike price
    T : float
        Time to expiration in years
    r : float
        Risk-free interest rate
    sigma : float
        Volatility (annualized)
    option_type : str, default='call'
        'call' or 'put'
    
    Returns
    -------
    dict
        Dictionary with keys: 'delta', 'gamma', 'theta', 'vega', 'rho'
    
    Examples
    --------
    >>> greeks = black_scholes_greeks(100, 105, 0.25, 0.05, 0.25, 'call')
    >>> greeks['delta']
    0.4297...
    """
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    
    if option_type == 'call':
        delta = norm.cdf(d1)
        theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) 
                 - r * K * np.exp(-r * T) * norm.cdf(d2)) / 365
    else:
        delta = -norm.cdf(-d1)
        theta = (-(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) 
                 + r * K * np.exp(-r * T) * norm.cdf(-d2)) / 365
    
    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))
    vega = S * norm.pdf(d1) * np.sqrt(T) / 100  # Per 1% change in volatility
    rho = K * T * np.exp(-r * T) * norm.cdf(d2) / 100 if option_type == 'call' else \
          -K * T * np.exp(-r * T) * norm.cdf(-d2) / 100
    
    return {
        'delta': delta,
        'gamma': gamma,
        'theta': theta,
        'vega': vega,
        'rho': rho
    }


def covered_call_payoff(S, S0, K, premium):
    """
    Calculate covered call strategy payoff.
    
    Parameters
    ----------
    S : np.ndarray
        Array of stock prices at expiration
    S0 : float
        Initial stock price (purchase price)
    K : float
        Call strike price
    premium : float
        Premium received from selling call
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'payoff': np.ndarray of payoffs
        - 'max_profit': float
        - 'max_loss': float
        - 'breakeven': float
    
    Examples
    --------
    >>> S = np.linspace(150, 200, 50)
    >>> result = covered_call_payoff(S, S0=175, K=180, premium=3.50)
    >>> result['max_profit']
    8.5
    """
    stock_pnl = S - S0
    call_pnl = -np.maximum(S - K, 0) + premium
    total_payoff = stock_pnl + call_pnl
    
    max_profit = K - S0 + premium
    max_loss = -S0 + premium
    breakeven = S0 - premium
    
    return {
        'payoff': total_payoff,
        'max_profit': max_profit,
        'max_loss': max_loss,
        'breakeven': breakeven,
        'stock_pnl': stock_pnl,
        'call_pnl': call_pnl
    }


def calculate_covered_call_greeks(S0, K, T, r, sigma):
    """
    Calculate Greeks for covered call strategy.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    K : float
        Call strike price
    T : float
        Time to expiration in years
    r : float
        Risk-free rate
    sigma : float
        Volatility
    
    Returns
    -------
    dict
        Greeks for the covered call position
    
    Examples
    --------
    >>> greeks = calculate_covered_call_greeks(175, 180, 0.0833, 0.05, 0.25)
    >>> greeks['delta']
    0.56...
    """
    call_greeks = black_scholes_greeks(S0, K, T, r, sigma, 'call')
    
    # Covered call = Long stock + Short call
    return {
        'delta': 1 - call_greeks['delta'],
        'gamma': -call_greeks['gamma'],
        'theta': -call_greeks['theta'],
        'vega': -call_greeks['vega'],
        'rho': -call_greeks['rho']
    }

print('[OK] Covered call functions implemented')
print('    - option_payoff(): Generic option payoff calculator')
print('    - black_scholes_greeks(): Greeks calculation')
print('    - covered_call_payoff(): Strategy payoff and metrics')
print('    - calculate_covered_call_greeks(): Strategy Greeks')

In [ ]:
# Visualization for Covered Call

# Parameters for AAPL covered call
S0 = 175.0  # Current AAPL price
K = 180.0   # Strike price
premium = 3.50  # Premium received
T = 30/365  # 30 days to expiration
r = 0.05    # Risk-free rate
sigma = 0.25  # Implied volatility

# Stock price range
S_range = np.linspace(150, 200, 200)

# Calculate payoff
result = covered_call_payoff(S_range, S0, K, premium)

# Calculate Greeks
greeks = calculate_covered_call_greeks(S0, K, T, r, sigma)

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Payoff diagram with components
ax1 = axes[0, 0]
ax1.plot(S_range, result['stock_pnl'], '--', label='Long Stock Only', linewidth=2, alpha=0.7)
ax1.plot(S_range, result['call_pnl'], '--', label='Short Call Only', linewidth=2, alpha=0.7)
ax1.plot(S_range, result['payoff'], linewidth=3, label='Covered Call', color=colors[0])
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axvline(x=S0, color='gray', linestyle=':', linewidth=1, alpha=0.5, label=f'Current Price (${S0})')
ax1.axvline(x=K, color='red', linestyle=':', linewidth=1, alpha=0.5, label=f'Strike (${K})')
ax1.axvline(x=result['breakeven'], color='green', linestyle=':', linewidth=1, alpha=0.5, 
            label=f'Breakeven (${result["breakeven"]:.2f})')
ax1.fill_between(S_range, 0, result['payoff'], where=(result['payoff'] > 0), 
                  alpha=0.2, color='green', label='Profit Zone')
ax1.fill_between(S_range, 0, result['payoff'], where=(result['payoff'] < 0), 
                  alpha=0.2, color='red', label='Loss Zone')
ax1.set_xlabel('Stock Price at Expiration ($)')
ax1.set_ylabel('Profit / Loss ($)')
ax1.set_title('Covered Call Payoff Diagram', fontweight='bold', fontsize=12)
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: P&L vs Stock Price with annotations
ax2 = axes[0, 1]
ax2.plot(S_range, result['payoff'], linewidth=3, color=colors[1])
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axhline(y=result['max_profit'], color='green', linestyle='--', linewidth=1, 
            label=f'Max Profit = ${result["max_profit"]:.2f}')
ax2.axvline(x=S0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
# Add annotations
ax2.annotate(f'Max Profit: ${result["max_profit"]:.2f}', 
             xy=(K, result['max_profit']), xytext=(K+5, result['max_profit']+2),
             arrowprops=dict(arrowstyle='->', color='green', lw=1.5),
             fontsize=9, fontweight='bold', color='green')
ax2.annotate(f'Breakeven: ${result["breakeven"]:.2f}', 
             xy=(result['breakeven'], 0), xytext=(result['breakeven']-10, -5),
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
             fontsize=9, fontweight='bold', color='blue')
ax2.set_xlabel('Stock Price at Expiration ($)')
ax2.set_ylabel('Profit / Loss ($)')
ax2.set_title('Covered Call P&L Analysis', fontweight='bold', fontsize=12)
ax2.legend(loc='best', fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Time decay impact (theta)
ax3 = axes[1, 0]
days_to_exp = np.array([30, 20, 10, 5, 1])
time_decay_values = []
for days in days_to_exp:
    T_temp = days / 365
    g = calculate_covered_call_greeks(S0, K, T_temp, r, sigma)
    time_decay_values.append(g['theta'])

ax3.plot(days_to_exp, time_decay_values, marker='o', linewidth=2, markersize=8, color=colors[2])
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.set_xlabel('Days to Expiration')
ax3.set_ylabel('Theta ($/day)')
ax3.set_title('Time Decay (Theta) Over Time', fontweight='bold', fontsize=12)
ax3.grid(True, alpha=0.3)
ax3.invert_xaxis()
for i, (day, theta_val) in enumerate(zip(days_to_exp, time_decay_values)):
    ax3.annotate(f'${theta_val:.3f}', xy=(day, theta_val), 
                xytext=(0, 10), textcoords='offset points', 
                ha='center', fontsize=8)

# Plot 4: Greeks summary
ax4 = axes[1, 1]
greek_names = ['Delta', 'Gamma', 'Theta', 'Vega']
greek_values = [greeks['delta'], greeks['gamma']*100, greeks['theta'], greeks['vega']]
colors_greeks = [colors[0], colors[1], colors[2], colors[3]]

bars = ax4.barh(greek_names, greek_values, color=colors_greeks, alpha=0.7, edgecolor='black')
ax4.axvline(x=0, color='black', linewidth=0.5)
ax4.set_xlabel('Greek Value')
ax4.set_title('Strategy Greeks Summary', fontweight='bold', fontsize=12)
ax4.grid(True, alpha=0.3, axis='x')

# Add value labels on bars
for i, (bar, value) in enumerate(zip(bars, greek_values)):
    width = bar.get_width()
    label_x_pos = width + (0.05 if width > 0 else -0.05)
    ax4.text(label_x_pos, bar.get_y() + bar.get_height()/2, f'{value:.3f}',
             ha='left' if width > 0 else 'right', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Covered Call Strategy: Comprehensive Analysis', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f'\nCovered Call Analysis Summary:')
print(f'{"="*50}')
print(f'Initial Stock Price: ${S0:.2f}')
print(f'Call Strike Price: ${K:.2f}')
print(f'Premium Received: ${premium:.2f}')
print(f'Maximum Profit: ${result["max_profit"]:.2f} ({result["max_profit"]/S0*100:.2f}% return)')
print(f'Breakeven Price: ${result["breakeven"]:.2f}')
print(f'Downside Cushion: ${premium:.2f} ({premium/S0*100:.2f}%)')
print(f'\nGreeks:')
print(f'  Delta: {greeks["delta"]:.3f} (exposure to price moves)')
print(f'  Theta: ${greeks["theta"]:.3f}/day (time decay benefit)')
print(f'  Vega: ${greeks["vega"]:.3f} (volatility exposure)')
print(f'{"="*50}')

#### Practical Example: AAPL Covered Call

**Scenario**: You own 100 shares of AAPL at $175/share and want to generate additional income. You decide to sell a covered call.

**Trade Setup:**
- Current AAPL price: $175
- Sell 1 ATM call option with strike $180 (slightly OTM)
- Expiration: 30 days
- Premium received: $3.50 per share ($350 for 100-share contract)
- Implied volatility: 25%

**Analysis of Outcomes:**

1. **Stock below $180 at expiration** (most likely scenario):
   - Keep the stock and the $350 premium
   - Can sell another call next month (generate recurring income)
   - Effective yield: $350 / $17,500 = 2% in 30 days (24% annualized)

2. **Stock above $180 at expiration**:
   - Stock called away at $180
   - Total profit: ($180 - $175) * 100 + $350 = $850
   - Return: $850 / $17,500 = 4.86% in 30 days
   - Downside: Miss out on gains above $180

3. **Stock drops significantly**:
   - Keep stock and premium
   - $350 premium provides 2% downside cushion
   - Breakeven: $171.50 (can lose $3.50 before actual loss)

**Risk Management Considerations:**
- Assignment risk: If stock rises above $180, be prepared to sell
- Rolling strategy: Can buy back call and sell further dated call if needed
- Downside exposure: Still have full downside risk minus premium
- Best when: Neutral to slightly bullish, elevated IV, willing to sell stock

**Real-World Results (Historical Data):**
- Win rate: ~65-70% of covered calls expire worthless or profitable
- Average return: 1-3% per month on the covered position
- Main risk: Opportunity cost when stock surges unexpectedly

In [ ]:
# Practical example calculation for Covered Call

# Define position parameters
shares = 100
stock_price = 175.0
call_strike = 180.0
premium_per_share = 3.50
position_value = shares * stock_price
premium_total = shares * premium_per_share

# Scenario analysis at different stock prices
scenarios = {
    'Bear Case ($165)': 165,
    'Base Case ($175)': 175,
    'Moderate Bull ($180)': 180,
    'Strong Bull ($190)': 190
}

print('Covered Call Scenario Analysis')
print('=' * 70)
print(f'Position: {shares} shares @ ${stock_price:.2f} = ${position_value:,.2f}')
print(f'Call Sold: {call_strike} strike @ ${premium_per_share:.2f} premium')
print(f'Total Premium Received: ${premium_total:.2f}')
print(f'Breakeven: ${stock_price - premium_per_share:.2f}')
print('=' * 70)

results_df = pd.DataFrame(columns=['Scenario', 'Final Price', 'Stock P&L', 'Call P&L', 
                                    'Total P&L', 'Return %'])

for scenario_name, final_price in scenarios.items():
    stock_pnl = (final_price - stock_price) * shares
    call_pnl = -max(final_price - call_strike, 0) * shares + premium_total
    total_pnl = stock_pnl + call_pnl
    return_pct = (total_pnl / position_value) * 100
    
    results_df = pd.concat([results_df, pd.DataFrame({
        'Scenario': [scenario_name],
        'Final Price': [f'${final_price:.2f}'],
        'Stock P&L': [f'${stock_pnl:.2f}'],
        'Call P&L': [f'${call_pnl:.2f}'],
        'Total P&L': [f'${total_pnl:.2f}'],
        'Return %': [f'{return_pct:.2f}%']
    })], ignore_index=True)

print('\nScenario Analysis:')
print(results_df.to_string(index=False))

# Calculate key metrics
max_profit = (call_strike - stock_price) * shares + premium_total
max_profit_pct = (max_profit / position_value) * 100
downside_protection = premium_per_share

print(f'\n\nKey Metrics:')
print(f'  Maximum Profit: ${max_profit:.2f} ({max_profit_pct:.2f}%)')
print(f'  Maximum Loss: Unlimited downside (minus ${premium_total:.2f} cushion)')
print(f'  Downside Protection: ${downside_protection:.2f} per share')
print(f'  Probability of Profit: ~70% (if stock stays above ${stock_price - premium_per_share:.2f})')
print(f'  Best Case: Stock finishes at ${call_strike:.2f} (max profit realized)')
print(f'  Worst Case: Stock drops significantly (still keep premium)')
print('=' * 70)

### 3. Protective Put Strategy

#### Theory and Market Context

The **protective put** (also called a married put) combines a long stock position with a long put option, providing **portfolio insurance** against downside risk. This strategy is used by investors who:

- **Own stock** and want downside protection without selling
- Are **concerned about short-term volatility** or market corrections
- Want to **protect unrealized gains** in appreciated positions
- Need **defined maximum loss** for risk management purposes

**Key Characteristics:**
- **Maximum Profit**: Unlimited (as stock rises)
- **Maximum Loss**: Limited to (S0 - K) + premium paid
- **Breakeven**: Stock price + premium paid
- **Market View**: Bullish long-term, bearish/uncertain short-term
- **Volatility Impact**: Benefits from increasing volatility (long option position)

**When to Use:**
1. Protect portfolio before earnings, events, or market uncertainty
2. Hold concentrated positions that need hedging
3. Lock in profits on appreciated stock without selling
4. Required risk management for institutional portfolios

**Risks:**
- Cost of protection (premium paid) reduces overall return
- Time decay works against you (theta negative)
- If market doesn't decline, premium is "wasted"
- Still exposed to gap risk beyond put protection

**Comparison to Other Protection Methods:**
- vs. Stop Loss: Guaranteed floor price, no gap risk
- vs. Collar: Lower cost but caps upside
- vs. Selling Stock: Keeps position, maintains upside exposure

#### Mathematical Formulation

The protective put payoff combines long stock and long put positions:

**Position Components:**
$$
\begin{align}
\text{Long Stock Position:} \quad & \Pi_{\text{stock}}(S_T) = S_T - S_0 \\
\text{Long Put Position:} \quad & \Pi_{\text{put}}(S_T) = \max(K - S_T, 0) - P_0
\end{align}
$$

**Total Payoff:**
$$
\Pi_{\text{protective put}}(S_T) = (S_T - S_0) + \max(K - S_T, 0) - P_0
$$

**Simplified Form:**
$$
\Pi_{\text{protective put}}(S_T) = \begin{cases}
K - S_0 - P_0 & \text{if } S_T < K \\
S_T - S_0 - P_0 & \text{if } S_T \geq K
\end{cases}
$$

**Key Metrics:**

Maximum Profit:
$$
\Pi_{\max} = \infty \quad \text{(unlimited as stock rises)}
$$

Maximum Loss:
$$
\Pi_{\min} = K - S_0 - P_0
$$
This is the guaranteed floor - you cannot lose more than this amount.

Breakeven Point:
$$
S_T^* = S_0 + P_0
$$

**Insurance Cost Analysis:**

Cost of Protection (as % of position):
$$
\text{Insurance Cost} = \frac{P_0}{S_0} \times 100\%
$$

Effective Protection Level:
$$
\text{Protection} = \frac{S_0 - K}{S_0} \times 100\%
$$

**Greeks for Protective Put:**

Delta (price sensitivity):
$$
\Delta_{\text{protective put}} = 1 + \Delta_{\text{put}} \approx 1 - N(d_1)
$$

Theta (time decay):
$$
\Theta_{\text{protective put}} = \Theta_{\text{put}} < 0
$$
The position has negative theta, losing value from time decay.

Vega (volatility sensitivity):
$$
\nu_{\text{protective put}} = \nu_{\text{put}} > 0
$$
The position benefits from increasing implied volatility.

Gamma (convexity):
$$
\Gamma_{\text{protective put}} = \Gamma_{\text{put}} > 0
$$
Positive convexity provides asymmetric protection.

where:
- $S_0$ = initial stock price
- $S_T$ = stock price at expiration  
- $K$ = put strike price (protection level)
- $P_0$ = put premium paid
- $N(d_1)$ = cumulative normal distribution from Black-Scholes

In [ ]:
# Python implementation for Protective Put

def protective_put_payoff(S, S0, K, premium):
    """
    Calculate protective put strategy payoff.
    
    Parameters
    ----------
    S : np.ndarray
        Array of stock prices at expiration
    S0 : float
        Initial stock price
    K : float
        Put strike price (protection level)
    premium : float
        Premium paid for put option
    
    Returns
    -------
    dict
        Dictionary containing:
        - 'payoff': np.ndarray of payoffs
        - 'max_profit': str (unlimited)
        - 'max_loss': float
        - 'breakeven': float
        - 'protection_level': float
    
    Examples
    --------
    >>> S = np.linspace(350, 450, 100)
    >>> result = protective_put_payoff(S, S0=400, K=380, premium=5.0)
    >>> result['max_loss']
    -25.0
    """
    stock_pnl = S - S0
    put_pnl = np.maximum(K - S, 0) - premium
    total_payoff = stock_pnl + put_pnl
    
    max_loss = K - S0 - premium
    breakeven = S0 + premium
    protection_level = S0 - K
    
    return {
        'payoff': total_payoff,
        'max_profit': 'Unlimited',
        'max_loss': max_loss,
        'breakeven': breakeven,
        'protection_level': protection_level,
        'stock_pnl': stock_pnl,
        'put_pnl': put_pnl,
        'insurance_cost_pct': (premium / S0) * 100
    }


def calculate_protective_put_greeks(S0, K, T, r, sigma):
    """
    Calculate Greeks for protective put strategy.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    K : float
        Put strike price
    T : float
        Time to expiration in years
    r : float
        Risk-free rate
    sigma : float
        Volatility
    
    Returns
    -------
    dict
        Greeks for the protective put position
    
    Examples
    --------
    >>> greeks = calculate_protective_put_greeks(400, 380, 0.0833, 0.05, 0.30)
    >>> greeks['delta']
    0.95...
    """
    put_greeks = black_scholes_greeks(S0, K, T, r, sigma, 'put')
    
    # Protective put = Long stock + Long put
    return {
        'delta': 1 + put_greeks['delta'],  # Long stock delta (1) + put delta (negative)
        'gamma': put_greeks['gamma'],
        'theta': put_greeks['theta'],
        'vega': put_greeks['vega'],
        'rho': put_greeks['rho']
    }


def compare_protection_costs(S0, strikes, T, r, sigma):
    """
    Compare costs of different protection levels.
    
    Parameters
    ----------
    S0 : float
        Current stock price
    strikes : list
        List of put strike prices
    T : float
        Time to expiration
    r : float
        Risk-free rate
    sigma : float
        Volatility
    
    Returns
    -------
    pd.DataFrame
        Comparison of protection costs and levels
    
    Examples
    --------
    >>> costs = compare_protection_costs(400, [360, 380, 390], 0.25, 0.05, 0.30)
    """
    from scipy.stats import norm
    
    results = []
    for K in strikes:
        # Black-Scholes put price
        d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
        d2 = d1 - sigma*np.sqrt(T)
        put_price = K*np.exp(-r*T)*norm.cdf(-d2) - S0*norm.cdf(-d1)
        
        protection_pct = ((S0 - K) / S0) * 100
        cost_pct = (put_price / S0) * 100
        
        results.append({
            'Strike': K,
            'Protection Level': f'{protection_pct:.1f}%',
            'Put Premium': f'${put_price:.2f}',
            'Cost %': f'{cost_pct:.2f}%',
            'Max Loss': f'${S0 - K + put_price:.2f}',
            'Breakeven': f'${S0 + put_price:.2f}'
        })
    
    return pd.DataFrame(results)

print('[OK] Protective put functions implemented')
print('    - protective_put_payoff(): Strategy payoff calculation')
print('    - calculate_protective_put_greeks(): Greeks for protective put')
print('    - compare_protection_costs(): Compare different strike prices')

In [ ]:
# Visualization for Protective Put

# Parameters for SPY portfolio protection
S0 = 400.0  # Current SPY price
portfolio_value = 100000  # $100k portfolio
shares = portfolio_value / S0
K = 380.0  # 5% OTM put for protection
premium = 5.00  # Put premium per share
T = 90/365  # 90 days to expiration
r = 0.05
sigma = 0.30  # Higher vol reflects market uncertainty

# Stock price range
S_range = np.linspace(340, 460, 200)

# Calculate payoff
result = protective_put_payoff(S_range, S0, K, premium)

# Calculate Greeks
greeks = calculate_protective_put_greeks(S0, K, T, r, sigma)

# Create 2x2 subplot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Payoff diagram comparison
ax1 = axes[0, 0]
naked_stock_pnl = S_range - S0
ax1.plot(S_range, naked_stock_pnl, '--', label='Unprotected Stock', linewidth=2, alpha=0.7, color=colors[4])
ax1.plot(S_range, result['payoff'], linewidth=3, label='Protected Stock (with Put)', color=colors[0])
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axhline(y=result['max_loss'], color='red', linestyle='--', linewidth=2, 
            label=f'Max Loss Floor: ${result["max_loss"]:.2f}')
ax1.axvline(x=S0, color='gray', linestyle=':', linewidth=1, alpha=0.5)
ax1.axvline(x=K, color='red', linestyle=':', linewidth=1, alpha=0.5, label=f'Protection Strike: ${K}')
ax1.fill_between(S_range, result['max_loss'], result['payoff'], 
                  where=(S_range < K), alpha=0.3, color='green', label='Insurance Benefit')
ax1.set_xlabel('Stock Price at Expiration ($)')
ax1.set_ylabel('Profit / Loss per Share ($)')
ax1.set_title('Protective Put vs. Naked Stock Position', fontweight='bold', fontsize=12)
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)

# Plot 2: Insurance cost analysis
ax2 = axes[0, 1]
strike_prices = np.arange(360, 400, 5)
protection_comparison = compare_protection_costs(S0, strike_prices, T, r, sigma)

# Extract numeric values for plotting
costs = [float(row['Cost %'].strip('%')) for _, row in protection_comparison.iterrows()]
protection_levels = [float(row['Protection Level'].strip('%')) for _, row in protection_comparison.iterrows()]

ax2.scatter(protection_levels, costs, s=100, alpha=0.6, c=colors[1], edgecolors='black', linewidth=1.5)
ax2.plot(protection_levels, costs, linewidth=2, alpha=0.5, color=colors[1])
# Highlight our chosen strike
chosen_idx = list(strike_prices).index(K)
ax2.scatter([protection_levels[chosen_idx]], [costs[chosen_idx]], 
           s=300, marker='*', color='red', edgecolors='black', linewidth=2, 
           label=f'Chosen: {protection_levels[chosen_idx]:.1f}% protection', zorder=5)
ax2.set_xlabel('Protection Level (% below current price)')
ax2.set_ylabel('Insurance Cost (% of position value)')
ax2.set_title('Cost vs. Protection Level Analysis', fontweight='bold', fontsize=12)
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)

# Add annotations
for i, (prot, cost) in enumerate(zip(protection_levels, costs)):
    if i % 2 == 0:  # Annotate every other point
        ax2.annotate(f'${strike_prices[i]}', xy=(prot, cost), 
                    xytext=(5, 5), textcoords='offset points', fontsize=7, alpha=0.7)

# Plot 3: Portfolio value protection
ax3 = axes[1, 0]
portfolio_unprotected = (S_range - S0) * shares
portfolio_protected = result['payoff'] * shares
ax3.plot(S_range, portfolio_unprotected, '--', label='Unprotected Portfolio', 
         linewidth=2, alpha=0.7, color=colors[4])
ax3.plot(S_range, portfolio_protected, linewidth=3, label='Protected Portfolio', color=colors[2])
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.axhline(y=result['max_loss']*shares, color='red', linestyle='--', linewidth=2,
            label=f'Maximum Loss: ${result["max_loss"]*shares:,.0f}')
ax3.fill_between(S_range, result['max_loss']*shares, 0, 
                  where=(portfolio_protected > result['max_loss']*shares),
                  alpha=0.2, color='red', label='Potential Unprotected Loss')
ax3.set_xlabel('Stock Price at Expiration ($)')
ax3.set_ylabel('Portfolio P&L ($)')
ax3.set_title(f'${portfolio_value:,.0f} Portfolio Protection', fontweight='bold', fontsize=12)
ax3.legend(loc='best', fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x/1000:.0f}k'))

# Plot 4: Greeks summary
ax4 = axes[1, 1]
greek_names = ['Delta', 'Gamma', 'Theta', 'Vega']
greek_values = [greeks['delta'], greeks['gamma']*100, greeks['theta'], greeks['vega']]
colors_greeks = [colors[0], colors[1], colors[2], colors[3]]

bars = ax4.barh(greek_names, greek_values, color=colors_greeks, alpha=0.7, edgecolor='black')
ax4.axvline(x=0, color='black', linewidth=0.5)
ax4.set_xlabel('Greek Value')
ax4.set_title('Strategy Greeks Summary', fontweight='bold', fontsize=12)
ax4.grid(True, alpha=0.3, axis='x')

for i, (bar, value) in enumerate(zip(bars, greek_values)):
    width = bar.get_width()
    label_x_pos = width + (0.02 if width > 0 else -0.02)
    ax4.text(label_x_pos, bar.get_y() + bar.get_height()/2, f'{value:.3f}',
             ha='left' if width > 0 else 'right', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Protective Put Strategy: Portfolio Insurance Analysis', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f'\nProtective Put Analysis Summary:')
print(f'{"="*60}')
print(f'Portfolio Value: ${portfolio_value:,.0f}')
print(f'Current SPY Price: ${S0:.2f} ({shares:.0f} shares)')
print(f'Protection Strike: ${K:.2f} ({(S0-K)/S0*100:.1f}% below current)')
print(f'Premium Paid: ${premium:.2f} per share (${premium*shares:,.0f} total)')
print(f'Insurance Cost: {result["insurance_cost_pct"]:.2f}% of portfolio value')
print(f'\nProtection Analysis:')
print(f'  Maximum Loss: ${result["max_loss"]*shares:,.0f} ({result["max_loss"]/S0*100:.2f}%)')
print(f'  Breakeven Price: ${result["breakeven"]:.2f} (+{(result["breakeven"]-S0)/S0*100:.2f}%)')
print(f'  Protected Below: ${K:.2f}')
print(f'\nGreeks (Portfolio Level):')
print(f'  Delta: {greeks["delta"]:.3f} (still ~100% exposure to upside)')
print(f'  Theta: ${greeks["theta"]*shares:,.2f}/day (cost of time decay)')
print(f'  Vega: ${greeks["vega"]*shares:,.2f} per 1% vol change')
print(f'{"="*60}')

#### Practical Example: Portfolio Protection for Market Uncertainty

**Scenario**: Portfolio manager with $100,000 in SPY holdings faces potential market correction.

**Trade Setup:**
- Current portfolio value: $100,000 (250 shares of SPY at $400)
- Market uncertainty due to Fed meeting and earnings season
- Purchase 3-month (90-day) protective puts
- Strike selection: $380 (5% below current, meaningful protection)
- Put premium: $5.00 per share ($1,250 total cost)

**Cost-Benefit Analysis:**

Protection costs $1,250 (1.25% of portfolio) for 90 days:
- Annualized insurance cost: 5% per year
- Maximum portfolio loss: $5,000 + $1,250 = $6,250 (6.25%)
- Protection kicks in if SPY drops below $380

**Historical Context:**
During the March 2020 COVID crash, SPY dropped from $340 to $220 (-35%) in 3 weeks. A protective put would have:
- Limited loss to strike price protection level
- Provided defined maximum loss regardless of crash severity
- Allowed keeping the position without forced selling

**Comparing Protection Levels:**

In [ ]:
# Practical example calculation - Protection comparison

# Compare different protection strike levels
S0 = 400.0
T = 90/365
r = 0.05
sigma = 0.30

strike_options = [360, 370, 380, 390, 395]
comparison_df = compare_protection_costs(S0, strike_options, T, r, sigma)

print('Protective Put Strike Comparison')
print('=' * 80)
print(comparison_df.to_string(index=False))
print('=' * 80)

# Detailed analysis for chosen strike ($380)
chosen_strike = 380
chosen_premium = 5.00
portfolio_value = 100000
shares = portfolio_value / S0

print(f'\\nChosen Protection: ${chosen_strike} Strike')
print(f'  Initial Position: ${portfolio_value:,.0f} ({shares:.0f} shares @ ${S0})')
print(f'  Protection Level: {(S0-chosen_strike)/S0*100:.1f}% below current price')
print(f'  Total Premium: ${chosen_premium * shares:,.0f}')
print(f'  Insurance Cost: {chosen_premium*shares/portfolio_value*100:.2f}% of portfolio')
print(f'  Max Portfolio Loss: ${(S0-chosen_strike+chosen_premium)*shares:,.0f}')
print(f'  Breakeven: ${S0 + chosen_premium:.2f} (need {chosen_premium/S0*100:.2f}% gain)')

# Stress test scenarios
print(f'\\nStress Test Scenarios:')
print(f'{"-"*60}')

scenarios = {
    'Mild Correction (-5%)': S0 * 0.95,
    'Moderate Drop (-10%)': S0 * 0.90,
    'Severe Drop (-20%)': S0 * 0.80,
    'Crash (-30%)': S0 * 0.70
}

for scenario_name, final_price in scenarios.items():
    unprotected_loss = (final_price - S0) * shares
    protected_loss = max((final_price - S0), (chosen_strike - S0)) * shares - chosen_premium * shares
    protection_benefit = unprotected_loss - protected_loss
    
    print(f'{scenario_name:30s} Final: ${final_price:.2f}')
    print(f'  {"Unprotected Loss:":30s} ${unprotected_loss:>10,.0f}')
    print(f'  {"Protected Loss:":30s} ${protected_loss:>10,.0f}')
    print(f'  {"Insurance Benefit:":30s} ${protection_benefit:>10,.0f}')
    print()

print('=' * 80)

### 4. Bull Call Spread Strategy

#### Theory and Market Context

The **bull call spread** is a limited-risk, limited-reward strategy combining a long call at lower strike (K1) and a short call at higher strike (K2). Used when:

- **Moderately bullish** - expect price increase but not dramatic rally
- **Cost reduction** - lower net premium than naked long call
- **Defined risk** - maximum loss is the net debit paid
- **High probability** - better breakeven than naked calls

**Key Characteristics:**
- **Maximum Profit**: (K2 - K1) - Net Debit
- **Maximum Loss**: Net Debit (premium paid)
- **Breakeven**: K1 + Net Debit
- **Market View**: Moderately bullish
- **Volatility Impact**: Near-neutral (long and short calls offset)

**When to Use:**
1. Bullish but want to cap cost and risk
2. Stock expected to rise moderately (not explosively)
3. Want better risk/reward than naked calls
4. Lower implied volatility makes spread attractive

**Advantages over Naked Long Call:**
- **Lower cost**: Short call premium offsets long call cost
- **Better breakeven**: Net debit lower than long call premium
- **Defined risk**: Can only lose net debit
- **Higher profit probability**: Lower breakeven point

**Trade-offs:**
- Capped upside (profit limited above K2)
- Two-leg execution (wider bid-ask)
- Early assignment risk on short call
- Miss out if stock surges above K2

#### Mathematical Formulation

The bull call spread combines long lower strike call with short higher strike call:

**Position Components:**
$$
\begin{align}
\text{Long Call (}K_1\text{):} \quad & \Pi_{\text{long}}(S_T) = \max(S_T - K_1, 0) - C_1 \\
\text{Short Call (}K_2\text{):} \quad & \Pi_{\text{short}}(S_T) = -\max(S_T - K_2, 0) + C_2
\end{align}
$$

where $K_1 < K_2$ (buy lower strike, sell higher strike) and $C_1 > C_2$ (lower strike costs more).

**Total Payoff:**
$$
\Pi_{\text{bull call}}(S_T) = \max(S_T - K_1, 0) - \max(S_T - K_2, 0) - (C_1 - C_2)
$$

**Simplified Piecewise Form:**
$$
\Pi_{\text{bull call}}(S_T) = \begin{cases}
-(C_1 - C_2) & \text{if } S_T < K_1 \quad \text{(both expire worthless)} \\
S_T - K_1 - (C_1 - C_2) & \text{if } K_1 \leq S_T < K_2 \quad \text{(long ITM, short OTM)} \\
K_2 - K_1 - (C_1 - C_2) & \text{if } S_T \geq K_2 \quad \text{(both ITM, offset)}
\end{cases}
$$

**Key Metrics:**

Net Debit (Initial Cost):
$$
\text{Debit} = C_1 - C_2
$$

Maximum Profit:
$$
\Pi_{\max} = K_2 - K_1 - (C_1 - C_2) = \text{Strike Width} - \text{Debit}
$$

Maximum Loss:
$$
\Pi_{\min} = -(C_1 - C_2) = -\text{Debit}
$$

Breakeven Point:
$$
S_T^* = K_1 + (C_1 - C_2) = K_1 + \text{Debit}
$$

**Risk/Reward Ratio:**
$$
\text{R/R Ratio} = \frac{\Pi_{\max}}{\text{Debit}} = \frac{K_2 - K_1 - \text{Debit}}{\text{Debit}}
$$

For a $5 wide spread costing $2: R/R = $(5-2)/2 = 1.5:1$

**Probability of Maximum Profit:**

Using log-normal distribution:
$$
P(S_T \geq K_2) = N(d_2^{K_2})
$$

where $d_2^{K_2} = \frac{\ln(S_0/K_2) + (r - 0.5\sigma^2)T}{\sigma\sqrt{T}}$

**Greeks for Bull Call Spread:**

Delta (directional exposure):
$$
\Delta_{\text{spread}} = \Delta_{K_1} - \Delta_{K_2} = N(d_1^{K_1}) - N(d_1^{K_2}) > 0
$$

Theta (time decay):
$$
\Theta_{\text{spread}} = \Theta_{K_1} - \Theta_{K_2}
$$

Vega (volatility sensitivity):
$$
\nu_{\text{spread}} = \nu_{K_1} - \nu_{K_2} \approx 0
$$
Near zero for equidistant strikes around spot (vega neutral).

Gamma (convexity):
$$
\Gamma_{\text{spread}} = \Gamma_{K_1} - \Gamma_{K_2}
$$

where:
- $K_1, K_2$ = lower and higher strikes ($K_1 < K_2$)
- $C_1, C_2$ = call premiums at respective strikes
- $S_T$ = stock price at expiration
- $N(\cdot)$ = cumulative standard normal distribution

In [ ]:
# Python implementation for Bull Call Spread and remaining strategies

def bull_call_spread_payoff(S, K1, K2, C1, C2):
    """
    Calculate bull call spread strategy payoff.
    
    Parameters
    ----------
    S : np.ndarray
        Array of stock prices at expiration
    K1 : float
        Lower strike price (long call)
    K2 : float
        Higher strike price (short call)
    C1 : float
        Premium paid for long call
    C2 : float
        Premium received for short call
    
    Returns
    -------
    dict
        Dictionary containing payoff metrics
    
    Examples
    --------
    >>> S = np.linspace(180, 240, 100)
    >>> result = bull_call_spread_payoff(S, K1=200, K2=220, C1=8.0, C2=3.0)
    >>> result['max_profit']
    15.0
    """
    long_call_pnl = np.maximum(S - K1, 0) - C1
    short_call_pnl = -np.maximum(S - K2, 0) + C2
    total_payoff = long_call_pnl + short_call_pnl
    
    net_debit = C1 - C2
    max_profit = K2 - K1 - net_debit
    max_loss = -net_debit
    breakeven = K1 + net_debit
    risk_reward = max_profit / net_debit if net_debit > 0 else 0
    
    return {
        'payoff': total_payoff,
        'max_profit': max_profit,
        'max_loss': max_loss,
        'breakeven': breakeven,
        'net_debit': net_debit,
        'risk_reward': risk_reward,
        'long_call_pnl': long_call_pnl,
        'short_call_pnl': short_call_pnl
    }


def bear_put_spread_payoff(S, K1, K2, P1, P2):
    """
    Calculate bear put spread payoff (buy higher strike, sell lower strike).
    
    Parameters
    ----------
    S : np.ndarray
        Stock prices at expiration
    K1 : float
        Lower strike (short put)
    K2 : float
        Higher strike (long put)
    P1 : float
        Premium received for short put
    P2 : float
        Premium paid for long put
    
    Returns
    -------
    dict
        Strategy payoff metrics
    """
    long_put_pnl = np.maximum(K2 - S, 0) - P2
    short_put_pnl = -np.maximum(K1 - S, 0) + P1
    total_payoff = long_put_pnl + short_put_pnl
    
    net_debit = P2 - P1
    max_profit = K2 - K1 - net_debit
    max_loss = -net_debit
    breakeven = K2 - net_debit
    
    return {
        'payoff': total_payoff,
        'max_profit': max_profit,
        'max_loss': max_loss,
        'breakeven': breakeven,
        'net_debit': net_debit,
        'long_put_pnl': long_put_pnl,
        'short_put_pnl': short_put_pnl
    }


def straddle_payoff(S, K, C_premium, P_premium):
    """
    Calculate long straddle payoff (long call + long put at same strike).
    
    Parameters
    ----------
    S : np.ndarray
        Stock prices at expiration
    K : float
        Strike price (same for call and put)
    C_premium : float
        Call premium paid
    P_premium : float
        Put premium paid
    
    Returns
    -------
    dict
        Strategy metrics
    """
    call_pnl = np.maximum(S - K, 0) - C_premium
    put_pnl = np.maximum(K - S, 0) - P_premium
    total_payoff = call_pnl + put_pnl
    
    total_premium = C_premium + P_premium
    breakeven_up = K + total_premium
    breakeven_down = K - total_premium
    
    return {
        'payoff': total_payoff,
        'max_loss': -total_premium,
        'breakeven_up': breakeven_up,
        'breakeven_down': breakeven_down,
        'total_premium': total_premium,
        'call_pnl': call_pnl,
        'put_pnl': put_pnl
    }


def strangle_payoff(S, K_put, K_call, P_premium, C_premium):
    """
    Calculate long strangle payoff (long OTM call + long OTM put).
    
    Parameters
    ----------
    S : np.ndarray
        Stock prices at expiration
    K_put : float
        Put strike (lower)
    K_call : float
        Call strike (higher)
    P_premium : float
        Put premium paid
    C_premium : float
        Call premium paid
    
    Returns
    -------
    dict
        Strategy metrics
    """
    call_pnl = np.maximum(S - K_call, 0) - C_premium
    put_pnl = np.maximum(K_put - S, 0) - P_premium
    total_payoff = call_pnl + put_pnl
    
    total_premium = C_premium + P_premium
    breakeven_up = K_call + total_premium
    breakeven_down = K_put - total_premium
    
    return {
        'payoff': total_payoff,
        'max_loss': -total_premium,
        'breakeven_up': breakeven_up,
        'breakeven_down': breakeven_down,
        'total_premium': total_premium,
        'call_pnl': call_pnl,
        'put_pnl': put_pnl
    }


def iron_condor_payoff(S, K_put_low, K_put_high, K_call_low, K_call_high,
                       P_low_premium, P_high_premium, C_low_premium, C_high_premium):
    """
    Calculate iron condor payoff (sell put spread + sell call spread).
    
    Parameters
    ----------
    S : np.ndarray
        Stock prices
    K_put_low, K_put_high : float
        Put spread strikes (sell high, buy low)
    K_call_low, K_call_high : float
        Call spread strikes (sell low, buy high)
    P_low_premium, P_high_premium : float
        Put premiums (pay low, receive high)
    C_low_premium, C_high_premium : float
        Call premiums (receive low, pay high)
    
    Returns
    -------
    dict
        Strategy metrics
    """
    # Short put spread
    put_spread_pnl = -np.maximum(K_put_high - S, 0) + P_high_premium + np.maximum(K_put_low - S, 0) - P_low_premium
    # Short call spread
    call_spread_pnl = -np.maximum(S - K_call_low, 0) + C_low_premium + np.maximum(S - K_call_high, 0) - C_high_premium
    
    total_payoff = put_spread_pnl + call_spread_pnl
    
    net_credit = (P_high_premium + C_low_premium) - (P_low_premium + C_high_premium)
    max_profit = net_credit
    max_loss = max(K_put_high - K_put_low, K_call_high - K_call_low) - net_credit
    
    return {
        'payoff': total_payoff,
        'max_profit': max_profit,
        'max_loss': -max_loss,
        'net_credit': net_credit,
        'profit_zone_low': K_put_high,
        'profit_zone_high': K_call_low
    }


def butterfly_spread_payoff(S, K1, K2, K3, C1, C2, C3):
    """
    Calculate butterfly spread (1 long low, 2 short mid, 1 long high).
    
    Parameters
    ----------
    S : np.ndarray
        Stock prices
    K1, K2, K3 : float
        Strikes (K1 < K2 < K3, equidistant)
    C1, C2, C3 : float
        Call premiums
    
    Returns
    -------
    dict
        Strategy metrics
    """
    long_low = np.maximum(S - K1, 0) - C1
    short_mid = -2 * (np.maximum(S - K2, 0) - C2)
    long_high = np.maximum(S - K3, 0) - C3
    
    total_payoff = long_low + short_mid + long_high
    
    net_debit = C1 - 2*C2 + C3
    max_profit = (K2 - K1) - net_debit
    max_loss = -net_debit
    breakeven_low = K1 + net_debit
    breakeven_high = K3 - net_debit
    
    return {
        'payoff': total_payoff,
        'max_profit': max_profit,
        'max_loss': max_loss,
        'breakeven_low': breakeven_low,
        'breakeven_high': breakeven_high,
        'net_debit': net_debit
    }

print('[OK] All spread strategy functions implemented')
print('    - bull_call_spread_payoff()')
print('    - bear_put_spread_payoff()')
print('    - straddle_payoff()')
print('    - strangle_payoff()')
print('    - iron_condor_payoff()')
print('    - butterfly_spread_payoff()')

In [ ]:
# Comprehensive visualization: Bull Call Spread with all major strategies comparison

# Bull Call Spread parameters - TSLA example
S0_tsla = 200.0
K1_bull = 200.0
K2_bull = 220.0
C1_bull = 12.0
C2_bull = 5.0

# Create comprehensive comparison
S_range = np.linspace(160, 260, 300)

# Calculate payoffs for different strategies
bull_call = bull_call_spread_payoff(S_range, K1_bull, K2_bull, C1_bull, C2_bull)
bear_put = bear_put_spread_payoff(S_range, K1=430, K2=450, P1=3.0, P2=8.0)  # Different asset
straddle = straddle_payoff(S_range, K=200, C_premium=12.0, P_premium=10.0)
strangle = strangle_payoff(S_range, K_put=190, K_call=210, P_premium=6.0, C_premium=7.0)

# Create 2x2 subplot for strategies
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Bull Call Spread detailed
ax1 = axes[0, 0]
ax1.plot(S_range, bull_call['long_call_pnl'], '--', label='Long 200 Call', linewidth=2, alpha=0.6)
ax1.plot(S_range, bull_call['short_call_pnl'], '--', label='Short 220 Call', linewidth=2, alpha=0.6)
ax1.plot(S_range, bull_call['payoff'], linewidth=3, label='Bull Call Spread', color=colors[0])
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axhline(y=bull_call['max_profit'], color='green', linestyle='--', linewidth=1, alpha=0.7)
ax1.axhline(y=bull_call['max_loss'], color='red', linestyle='--', linewidth=1, alpha=0.7)
ax1.axvline(x=S0_tsla, color='gray', linestyle=':', linewidth=1, alpha=0.5, label=f'Current: ${S0_tsla}')
ax1.axvline(x=bull_call['breakeven'], color='blue', linestyle=':', linewidth=1, alpha=0.5)
ax1.fill_between(S_range, 0, bull_call['payoff'], where=(bull_call['payoff'] > 0), 
                  alpha=0.2, color='green')
ax1.fill_between(S_range, 0, bull_call['payoff'], where=(bull_call['payoff'] < 0), 
                  alpha=0.2, color='red')
ax1.set_xlabel('TSLA Price at Expiration ($)')
ax1.set_ylabel('Profit / Loss ($)')
ax1.set_title('Bull Call Spread: TSLA $200/$220', fontweight='bold')
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)
ax1.set_ylim([-15, 25])

# Add annotations
ax1.annotate(f'Max Profit: ${bull_call["max_profit"]:.2f}',
             xy=(K2_bull, bull_call['max_profit']), xytext=(K2_bull+10, bull_call['max_profit']+3),
             arrowprops=dict(arrowstyle='->', color='green', lw=1.5),
             fontsize=9, fontweight='bold', color='green')
ax1.annotate(f'Max Loss: ${bull_call["max_loss"]:.2f}\\nRisk: ${bull_call["net_debit"]:.2f}',
             xy=(S_range[20], bull_call['max_loss']), xytext=(S_range[20], bull_call['max_loss']-5),
             fontsize=9, fontweight='bold', color='red')
ax1.annotate(f'BE: ${bull_call["breakeven"]:.2f}',
             xy=(bull_call['breakeven'], 0), xytext=(bull_call['breakeven']-5, -8),
             arrowprops=dict(arrowstyle='->', color='blue', lw=1.5),
             fontsize=9, fontweight='bold', color='blue')

# Plot 2: Bear Put Spread
ax2 = axes[0, 1]
ax2.plot(S_range + 230, bear_put['payoff'], linewidth=3, color=colors[1])  # Shift for SPY
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axhline(y=bear_put['max_profit'], color='green', linestyle='--', linewidth=1, alpha=0.7, 
            label=f'Max Profit: ${bear_put["max_profit"]:.2f}')
ax2.axhline(y=bear_put['max_loss'], color='red', linestyle='--', linewidth=1, alpha=0.7,
            label=f'Max Loss: ${bear_put["max_loss"]:.2f}')
ax2.axvline(x=440, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Current: $440')
ax2.fill_between(S_range+230, 0, bear_put['payoff'], where=(bear_put['payoff'] > 0), 
                  alpha=0.2, color='green')
ax2.fill_between(S_range+230, 0, bear_put['payoff'], where=(bear_put['payoff'] < 0), 
                  alpha=0.2, color='red')
ax2.set_xlabel('SPY Price at Expiration ($)')
ax2.set_ylabel('Profit / Loss ($)')
ax2.set_title('Bear Put Spread: SPY $450/$430', fontweight='bold')
ax2.legend(loc='best', fontsize=8)
ax2.grid(True, alpha=0.3)
ax2.set_xlim([390, 490])

# Plot 3: Straddle vs Strangle
ax3 = axes[1, 0]
ax3.plot(S_range, straddle['payoff'], linewidth=3, label='Straddle (ATM)', color=colors[2])
ax3.plot(S_range, strangle['payoff'], linewidth=3, label='Strangle (OTM)', color=colors[3], linestyle='--')
ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax3.axvline(x=S0_tsla, color='gray', linestyle=':', linewidth=1, alpha=0.5, label='Current Price')
# Mark breakevens
ax3.axvline(x=straddle['breakeven_up'], color=colors[2], linestyle=':', linewidth=1, alpha=0.5)
ax3.axvline(x=straddle['breakeven_down'], color=colors[2], linestyle=':', linewidth=1, alpha=0.5)
ax3.axvline(x=strangle['breakeven_up'], color=colors[3], linestyle=':', linewidth=1, alpha=0.5)
ax3.axvline(x=strangle['breakeven_down'], color=colors[3], linestyle=':', linewidth=1, alpha=0.5)
ax3.fill_between(S_range, straddle['max_loss'], straddle['payoff'], 
                  where=(straddle['payoff'] > straddle['max_loss']),
                  alpha=0.1, color=colors[2])
ax3.set_xlabel('Stock Price at Expiration ($)')
ax3.set_ylabel('Profit / Loss ($)')
ax3.set_title('Volatility Strategies: Straddle vs Strangle', fontweight='bold')
ax3.legend(loc='best', fontsize=8)
ax3.grid(True, alpha=0.3)

# Plot 4: Strategy Comparison Summary
ax4 = axes[1, 1]
strategies = ['Bull Call\\nSpread', 'Bear Put\\nSpread', 'Straddle', 'Strangle']
max_profits = [bull_call['max_profit'], bear_put['max_profit'], 'Unlimited', 'Unlimited']
max_losses = [abs(bull_call['max_loss']), abs(bear_put['max_loss']), 
              abs(straddle['max_loss']), abs(strangle['max_loss'])]
risk_rewards = [bull_call['risk_reward'], bear_put['max_profit']/abs(bear_put['max_loss']), 
                '-', '-']

y_pos = np.arange(len(strategies))
bars = ax4.barh(y_pos, [bull_call['max_profit'], bear_put['max_profit'], 30, 25], 
                color=colors[:4], alpha=0.7, edgecolor='black')
ax4.set_yticks(y_pos)
ax4.set_yticklabels(strategies)
ax4.set_xlabel('Maximum Profit ($)')
ax4.set_title('Strategy Comparison: Max Profit', fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (bar, profit) in enumerate(zip(bars, max_profits)):
    width = bar.get_width()
    label = f'${profit}' if isinstance(profit, (int, float)) else profit
    ax4.text(width + 1, bar.get_y() + bar.get_height()/2, label,
             ha='left', va='center', fontsize=9, fontweight='bold')

plt.suptitle('Option Strategies: Comprehensive Comparison', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

print(f'\\nBull Call Spread Analysis (TSLA $200/$220):')
print(f'{"="*60}')
print(f'Net Debit: ${bull_call["net_debit"]:.2f}')
print(f'Maximum Profit: ${bull_call["max_profit"]:.2f}')
print(f'Maximum Loss: ${bull_call["max_loss"]:.2f}')
print(f'Breakeven: ${bull_call["breakeven"]:.2f}')
print(f'Risk/Reward Ratio: {bull_call["risk_reward"]:.2f}:1')
print(f'Profit Zone: ${bull_call["breakeven"]:.2f} to ${K2_bull} and above')
print(f'{"="*60}')

#### Practical Example: TSLA Bull Call Spread

**Scenario**: Trader is bullish on TSLA but wants defined risk. TSLA currently at $200.

**Trade Setup:**
- Buy 1 TSLA $200 call @ $12.00
- Sell 1 TSLA $220 call @ $5.00
- Net debit: $7.00 per share ($700 per contract)
- Expiration: 45 days
- Implied volatility: 50% (elevated due to recent news)

**Analysis:**

- **Maximum Profit**: $(220 - 200) - $7 = $13.00 per share ($1,300 total)
- **Maximum Loss**: $7.00 per share ($700 total)
- **Breakeven**: $200 + $7 = $207.00
- **Risk/Reward**: $13/$7 = 1.86:1
- **Probability of Profit**: ~55% (stock needs to rise above $207)

**Comparison to Naked Long Call:**
- Naked $200 call costs $12.00 (vs $7.00 for spread)
- Spread has lower breakeven ($207 vs $212)
- But spread caps profit at $220 (naked call unlimited)
- Spread has 71% better risk/reward

**Best Case**: TSLA finishes at or above $220 - full $1,300 profit realized

**Worst Case**: TSLA drops below $200 - lose $700 (known max loss)

**At Breakeven**: TSLA at $207 - no profit, no loss

**Why This Works:**
- TSLA frequently moves 5-10% in 45 days
- $200 to $220 is 10% move (achievable)
- Defined risk allows position sizing based on capital
- Can allocate multiple spreads across different strikes/expirations

In [ ]:
# Calculate detailed bull call spread example with visualization summary

# Example trade details
print('Bull Call Spread Trade Example')
print('=' * 70)
print('\\nUnderlying: TSLA @ $200')
print('Strategy: Buy $200 Call, Sell $220 Call')
print('Expiration: 45 Days')
print('\\nEntry Prices:')
print('  Long $200 Call:  -$12.00')
print('  Short $220 Call: +$5.00')
print('  Net Debit:       -$7.00 per share')
print('\\nPer Contract (100 shares):')
print('  Capital Required: $700')
print('  Maximum Profit: $1,300 (at or above $220)')
print('  Maximum Loss: $700 (at or below $200)')
print('  Breakeven: $207.00')
print('  Risk/Reward: 1.86:1')
print('\\nPrice Targets:')
scenarios_bull = [
    ('Below $200', 200, -7.00),
    ('At $207 (BE)', 207, 0.00),
    ('At $210', 210, 3.00),
    ('At $220+', 220, 13.00)
]
for scenario, price, pnl in scenarios_bull:
    print(f'  {scenario:15s} ${price:>6.2f} → P&L: ${pnl:>6.2f} per share')

print('\\nWhen to Use:')
print('  ✓ Moderately bullish on TSLA')
print('  ✓ Want defined risk (not willing to lose more than $700)')
print('  ✓ Expect move to $220 but not much beyond')
print("  ✓ Don't want to pay $12 for naked call")
print('=' * 70)

### 5. Bear Put Spread Strategy

#### Theory and Market Context

The **bear put spread** is the bearish counterpart to the bull call spread, combining a long put at higher strike with a short put at lower strike. Used when:

- **Moderately bearish** outlook on underlying
- Want **defined risk** and lower cost than naked puts
- Expect price decline but not a crash
- **Reduce cost** of put protection by selling lower strike

**Key Characteristics:**
- **Maximum Profit**: (K2 - K1) - Net Debit
- **Maximum Loss**: Net Debit (premium paid)
- **Breakeven**: K2 - Net Debit  
- **Market View**: Moderately bearish
- **Volatility Impact**: Minimal (long and short puts offset)

**When to Use:**
1. Bearish on stock but want limited risk
2. Expect moderate decline, not catastrophic drop
3. Want better risk/reward than naked put
4. Volatility expected to remain stable

**Structure Example:**
- Buy $450 SPY put (higher strike, more expensive)
- Sell $430 SPY put (lower strike, cheaper)
- Net debit = P(450) - P(430)
- Profit if SPY declines below $450

**Advantages over Naked Put:**
- Lower cost (offset by short put premium)
- Defined maximum loss
- Still profitable on moderate declines
- Better capital efficiency

**Risks:**
- Limited profit potential (capped at lower strike)
- Still lose full debit if stock doesn't decline
- Short put can be assigned if breached
- Commissions on two-leg order

#### Mathematical Formulation

The mathematical framework for bear put spread is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Bear Put Spread

def compute_bear_put_spread():
    """
    Bear Put Spread implementation
    """
    # Implementation code here
    pass

print(f'[OK] Bear Put Spread implementation complete')

In [ ]:
# Visualization for Bear Put Spread

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Bear Put Spread')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply bear put spread to a real-world scenario...

In [ ]:
# Practical example for Bear Put Spread

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 6. Straddle and Strangle Strategies

#### Theory and Market Context

**Straddle** and **strangle** are volatility strategies that profit from large price movements in either direction. These strategies are employed when:

- **High volatility expected** but direction uncertain
- **Earnings announcements** or major events approaching  
- **Breaking news** that could move stock significantly
- **Market expects big move** (implied volatility elevated)

#### Straddle (ATM)

The **long straddle** buys both a call and put at the **same strike** (typically ATM).

**Key Characteristics:**
- **Maximum Loss**: Total premium paid (call + put)
- **Maximum Profit**: Unlimited in either direction
- **Breakeven Points**: Strike ± Total Premium (two breakevens)
- **Market View**: Expect large move, don't know direction
- **Cost**: Expensive due to two ATM options

**When to Use:**
1. Before earnings when large move expected
2. FDA announcements for biotech stocks
3. Fed meetings, major economic data
4. Takeover rumors or major corporate events

#### Strangle (OTM)

The **long strangle** buys an OTM call and OTM put.

**Key Characteristics:**
- **Maximum Loss**: Total premium (less than straddle)
- **Maximum Profit**: Unlimited in either direction
- **Breakeven Points**: Call strike + premium, Put strike - premium
- **Market View**: Expect very large move
- **Cost**: Cheaper than straddle (OTM options)

**Straddle vs Strangle:**

| Feature | Straddle | Strangle |
|---------|----------|----------|
| **Cost** | Higher (ATM options) | Lower (OTM options) |
| **Move Required** | Smaller | Larger |
| **Breakevens** | Closer | Wider |
| **Delta** | 0 (neutral) | 0 (neutral) |
| **Vega** | Higher (more vol exposure) | Lower |
| **Theta** | More negative | Less negative |

#### Risks and Considerations

**Volatility Crush:**
- IV often spikes before earnings, then collapses after
- Stock can move 5% but trade still loses money
- Need to account for IV crush in profit calculations

**Time Decay:**
- Both strategies have negative theta
- Every day that passes without move = loss
- Typically hold <30 days to expiration

**Probability:**
- Need stock to move >1 standard deviation
- Historical: stocks move >1σ about 32% of the time
- Pricing often assumes higher probability than reality

**Best Practices:**
1. Enter when IV is relatively low (IV percentile <50%)
2. Look for binary events with uncertain outcomes
3. Consider closing before event if IV spikes (sell vol)
4. Manage size - these can be expensive
5. Consider using spreads instead (iron butterfly) to cap losses

#### Mathematical Formulation

The mathematical framework for straddle and strangle is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Straddle and Strangle

def compute_straddle_and_strangle():
    """
    Straddle and Strangle implementation
    """
    # Implementation code here
    pass

print(f'[OK] Straddle and Strangle implementation complete')

In [ ]:
# Visualization for Straddle and Strangle

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Straddle and Strangle')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply straddle and strangle to a real-world scenario...

In [ ]:
# Practical example for Straddle and Strangle

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 7. Iron Condor Strategy

#### Theory and Market Context

The **iron condor** is a market-neutral, income-generating strategy that combines a **bear call spread** (above current price) and a **bull put spread** (below current price). This creates a "profit zone" where the trader profits if the stock stays within a defined range.

Structure:
1. Sell OTM put spread (buy lower strike, sell higher strike)
2. Sell OTM call spread (sell lower strike, buy higher strike)
3. Collect net premium (credit)

**Key Characteristics:**
- **Maximum Profit**: Net credit received
- **Maximum Loss**: Width of spread - net credit
- **Profit Zone**: Between the two short strikes
- **Market View**: Low volatility, range-bound market
- **Ideal Conditions**: High IV at entry, then vol decreases

**When to Use:**
1. Market in consolidation/range
2. After earnings (IV crush expected)
3. Stable market conditions
4. Want theta-positive strategy
5. High current IV (expensive options = better premium)

**Income Generation Strategy:**

Iron condors are the workhorse of options income traders:
- Probability of profit: 60-70% typically
- Target: Keep 50-75% of credit, close early
- Monthly strategy: 30-45 days to expiration
- Size multiple small positions, not one large

**Risk Profile:**
- Defined maximum loss on both sides
- Loss occurs if stock breaches either wing
- Early closing reduces tail risk
- Can be adjusted by rolling threatened side

**Advantages:**
- Predictable income in range-bound markets
- Positive theta (time decay works for you)
- Defined risk on both sides
- Benefits from decreasing volatility
- Lower margin requirements than naked options

**Disadvantages:**
- Limited profit potential
- Can lose much more than profit possible (risk/reward often 3:1 against you)
- Requires active management
- Commissions on 4-leg trade
- Can be difficult to adjust

**Trade Management:**
1. Enter with 30-45 DTE
2. Close at 50-75% max profit
3. Adjust if tested (roll, add hedge)
4. Don't hold to expiration unless deep profitable

#### Mathematical Formulation

The mathematical framework for iron condor is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Iron Condor

def compute_iron_condor():
    """
    Iron Condor implementation
    """
    # Implementation code here
    pass

print(f'[OK] Iron Condor implementation complete')

In [ ]:
# Comprehensive Visualization: Iron Condor and Advanced Strategies

# Iron Condor parameters - SPY example
S0_spy = 450.0
K_put_low = 440.0
K_put_high = 445.0
K_call_low = 455.0
K_call_high = 460.0

# Premiums (receive for short strikes, pay for long strikes)
P_low_prem = 1.00  # Pay for protection
P_high_prem = 2.50  # Receive for short put
C_low_prem = 2.40  # Receive for short call
C_high_prem = 0.90  # Pay for protection

# Stock price range
S_range_spy = np.linspace(430, 470, 300)

# Calculate payoffs
iron_condor = iron_condor_payoff(S_range_spy, K_put_low, K_put_high, K_call_low, K_call_high,
                                  P_low_prem, P_high_prem, C_low_prem, C_high_prem)

# Butterfly spread parameters
butterfly = butterfly_spread_payoff(S_range_spy, K1=445, K2=450, K3=455, 
                                    C1=7.50, C2=5.00, C3=3.00)

# Create 2x2 visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Iron Condor detailed payoff
ax1 = axes[0, 0]
ax1.plot(S_range_spy, iron_condor['payoff'], linewidth=3, color=colors[0], label='Iron Condor')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax1.axhline(y=iron_condor['max_profit'], color='green', linestyle='--', linewidth=2, alpha=0.7,
            label=f'Max Profit: ${iron_condor["max_profit"]:.2f}')
ax1.axhline(y=iron_condor['max_loss'], color='red', linestyle='--', linewidth=2, alpha=0.7,
            label=f'Max Loss: ${iron_condor["max_loss"]:.2f}')
ax1.axvline(x=S0_spy, color='gray', linestyle=':', linewidth=2, alpha=0.7, label=f'Current: ${S0_spy}')
ax1.axvline(x=K_put_high, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax1.axvline(x=K_call_low, color='red', linestyle=':', linewidth=1, alpha=0.5)

# Shade profit zone
ax1.fill_between(S_range_spy, iron_condor['max_loss'], iron_condor['payoff'],
                 where=((S_range_spy >= K_put_high) & (S_range_spy <= K_call_low)),
                 alpha=0.3, color='green', label='Profit Zone')
# Shade loss zones
ax1.fill_between(S_range_spy, iron_condor['max_loss'], iron_condor['payoff'],
                 where=(S_range_spy < K_put_high),
                 alpha=0.2, color='red')
ax1.fill_between(S_range_spy, iron_condor['max_loss'], iron_condor['payoff'],
                 where=(S_range_spy > K_call_low),
                 alpha=0.2, color='red')

ax1.set_xlabel('SPY Price at Expiration ($)')
ax1.set_ylabel('Profit / Loss ($)')
ax1.set_title('Iron Condor: SPY 440/445/455/460', fontweight='bold', fontsize=12)
ax1.legend(loc='best', fontsize=8)
ax1.grid(True, alpha=0.3)

# Annotations
ax1.annotate('Profit Zone', xy=(450, iron_condor['max_profit']), 
             xytext=(450, iron_condor['max_profit']+0.5),
             fontsize=10, ha='center', fontweight='bold', color='green')
ax1.annotate(f'Loss if\\nbelow ${K_put_high}', xy=(437, iron_condor['max_loss']),
             fontsize=9, ha='center', color='red')
ax1.annotate(f'Loss if\\nabove ${K_call_low}', xy=(463, iron_condor['max_loss']),
             fontsize=9, ha='center', color='red')

# Plot 2: Butterfly Spread
ax2 = axes[0, 1]
ax2.plot(S_range_spy, butterfly['payoff'], linewidth=3, color=colors[1])
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.axhline(y=butterfly['max_profit'], color='green', linestyle='--', linewidth=1, alpha=0.7,
            label=f'Max Profit: ${butterfly["max_profit"]:.2f}')
ax2.axvline(x=450, color='green', linestyle=':', linewidth=2, alpha=0.7, label='Peak at $450')
ax2.axvline(x=butterfly['breakeven_low'], color='blue', linestyle=':', linewidth=1, alpha=0.5,
            label=f'BE Low: ${butterfly["breakeven_low"]:.2f}')
ax2.axvline(x=butterfly['breakeven_high'], color='blue', linestyle=':', linewidth=1, alpha=0.5,
            label=f'BE High: ${butterfly["breakeven_high"]:.2f}')
ax2.fill_between(S_range_spy, butterfly['max_loss'], butterfly['payoff'],
                 where=(butterfly['payoff'] > butterfly['max_loss']),
                 alpha=0.2, color='green')
ax2.set_xlabel('Stock Price at Expiration ($)')
ax2.set_ylabel('Profit / Loss ($)')
ax2.set_title('Butterfly Spread: 445/450/455', fontweight='bold', fontsize=12)
ax2.legend(loc='best', fontsize=8)
ax2.grid(True, alpha=0.3)

# Plot 3: Probability of Profit zones
ax3 = axes[1, 0]
# Create probability distribution
from scipy.stats import norm as normal_dist
sigma_spy = 0.15  # 15% annualized vol
T_days = 45
T_years = T_days / 365
expected_move = S0_spy * sigma_spy * np.sqrt(T_years)

# Normal distribution centered at current price
x_prob = np.linspace(430, 470, 200)
prob_density = normal_dist.pdf(x_prob, S0_spy, expected_move)

ax3.plot(x_prob, prob_density, linewidth=2, color='purple', label='Price Distribution')
ax3.fill_between(x_prob, 0, prob_density, 
                 where=((x_prob >= K_put_high) & (x_prob <= K_call_low)),
                 alpha=0.3, color='green', label='Iron Condor Profit Zone')
ax3.axvline(x=S0_spy, color='black', linestyle='--', linewidth=2, label='Current Price')
ax3.axvline(x=K_put_high, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax3.axvline(x=K_call_low, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax3.set_xlabel('SPY Price ($)')
ax3.set_ylabel('Probability Density')
ax3.set_title(f'Price Distribution ({T_days} days, {sigma_spy*100:.0f}% vol)', fontweight='bold', fontsize=12)
ax3.legend(loc='best', fontsize=8)
ax3.grid(True, alpha=0.3)

# Calculate probability of profit
prob_profit = normal_dist.cdf(K_call_low, S0_spy, expected_move) - normal_dist.cdf(K_put_high, S0_spy, expected_move)
ax3.text(0.95, 0.95, f'P(Profit) ≈ {prob_profit*100:.1f}%', 
         transform=ax3.transAxes, fontsize=11, fontweight='bold',
         verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Plot 4: Strategy comparison summary
ax4 = axes[1, 1]
strategies_data = {
    'Iron\\nCondor': {'Max Loss': abs(iron_condor['max_loss']), 'Max Profit': iron_condor['max_profit'], 
                      'Complexity': 4},
    'Butterfly': {'Max Loss': abs(butterfly['max_loss']), 'Max Profit': butterfly['max_profit'],
                  'Complexity': 3},
    'Bull Call\\nSpread': {'Max Loss': 7, 'Max Profit': 13, 'Complexity': 2},
    'Covered\\nCall': {'Max Loss': 100, 'Max Profit': 8, 'Complexity': 1}
}

x_pos = np.arange(len(strategies_data))
max_losses = [data['Max Loss'] for data in strategies_data.values()]
max_profits = [data['Max Profit'] for data in strategies_data.values()]

width = 0.35
bars1 = ax4.bar(x_pos - width/2, max_profits, width, label='Max Profit', color='green', alpha=0.7)
bars2 = ax4.bar(x_pos + width/2, max_losses, width, label='Max Loss', color='red', alpha=0.7)

ax4.set_xlabel('Strategy')
ax4.set_ylabel('Amount ($)')
ax4.set_title('Risk/Reward Comparison', fontweight='bold', fontsize=12)
ax4.set_xticks(x_pos)
ax4.set_xticklabels(strategies_data.keys(), fontsize=9)
ax4.legend(loc='best', fontsize=9)
ax4.grid(True, alpha=0.3, axis='y')

plt.suptitle('Advanced Option Strategies: Iron Condor & Butterfly Analysis', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

# Summary statistics
net_credit = iron_condor['net_credit']
max_risk = abs(iron_condor['max_loss'])
risk_reward_ratio = max_risk / net_credit if net_credit > 0 else 0

print(f'\\nIron Condor Trade Analysis (SPY):')
print(f'{"="*70}')
print(f'Structure: {K_put_low}/{K_put_high}/{K_call_low}/{K_call_high}')
print(f'\\nPremiums:')
print(f'  Sell {K_put_high} Put: +${P_high_prem:.2f}')
print(f'  Buy {K_put_low} Put: -${P_low_prem:.2f}')
print(f'  Sell {K_call_low} Call: +${C_low_prem:.2f}')
print(f'  Buy {K_call_high} Call: -${C_high_prem:.2f}')
print(f'  Net Credit: ${net_credit:.2f}')
print(f'\\nRisk/Reward:')
print(f'  Maximum Profit: ${iron_condor["max_profit"]:.2f}')
print(f'  Maximum Loss: ${iron_condor["max_loss"]:.2f}')
print(f'  Risk/Reward Ratio: {risk_reward_ratio:.2f}:1')
print(f'  Profit Zone: ${K_put_high:.2f} to ${K_call_low:.2f}')
print(f'  Width of Profit Zone: ${K_call_low - K_put_high:.2f} ({(K_call_low-K_put_high)/S0_spy*100:.1f}%)')
print(f'  Probability of Profit: ~{prob_profit*100:.1f}%')
print(f'\\nReturn Metrics:')
print(f'  Return on Risk: {(net_credit/max_risk)*100:.1f}%')
print(f'  Days to Expiration: {T_days}')
print(f'  Annualized Return: ~{(net_credit/max_risk)*(365/T_days)*100:.1f}%')
print(f'{"="*70}')

#### Practical Example

Let's apply iron condor to a real-world scenario...

In [ ]:
# Practical example for Iron Condor

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 8. Butterfly Spread

### Theory

Detailed explanation of butterfly spread...

#### Mathematical Formulation

The mathematical framework for butterfly spread is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Butterfly Spread

def compute_butterfly_spread():
    """
    Butterfly Spread implementation
    """
    # Implementation code here
    pass

print(f'[OK] Butterfly Spread implementation complete')

In [ ]:
# Visualization for Butterfly Spread

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Butterfly Spread')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply butterfly spread to a real-world scenario...

In [ ]:
# Practical example for Butterfly Spread

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 9. Calendar Spread Strategy

#### Theory and Market Context

The **calendar spread** (also called time spread or horizontal spread) involves selling a near-term option and buying a longer-term option at the **same strike price**. This strategy profits from:

- **Time decay differential**: Near-term options decay faster than long-term
- **Stable price action**: Works best when price stays near strike
- **Volatility changes**: Benefits if IV increases for long-term option

**Key Characteristics:**
- **Maximum Profit**: Achieved when stock at strike at near expiration
- **Maximum Loss**: Net debit paid
- **Breakeven**: Complex (depends on vol and time)
- **Market View**: Neutral, expect sideways movement
- **Unique Feature**: Requires different expirations (can't calculate simple payoff at expiry)

**Structure:**
- Sell 30-day $100 call
- Buy 90-day $100 call
- Collect near-term premium
- Keep long-term exposure

**When to Use:**
1. Expect stock to trade near specific price
2. Anticipate volatility increase in future
3. Want to reduce cost of long-term option
4. Neutral to slightly bullish/bearish

**How It Works:**

Near expiration of short option:
- If stock near strike: Short expires worthless, long retains value
- Profit = Short premium - decay on long option
- Can sell another short option against same long (roll)

**Variants:**
- **Call Calendar**: Slightly bullish bias
- **Put Calendar**: Slightly bearish bias
- **Diagonal Spread**: Different strikes AND expirations

**Advantages:**
- Lower cost than outright long option
- Benefits from theta decay differential
- Can repeat (roll short option)
- Vega positive (benefits from vol increase)

**Disadvantages:**
- Requires Black-Scholes pricing (can't just use intrinsic value)
- Limited profit potential
- Complex P&L dynamics
- Early assignment risk on short option
- Need stock to stay near strike

#### Mathematical Formulation

The mathematical framework for calendar spread is given by:

$$\n\\text{Equation placeholder}\n$$

where the parameters represent key variables in the model.

In [ ]:
# Python implementation for Calendar Spread

def compute_calendar_spread():
    """
    Calendar Spread implementation
    """
    # Implementation code here
    pass

print(f'[OK] Calendar Spread implementation complete')

In [ ]:
# Visualization for Calendar Spread

fig, ax = plt.subplots(figsize=(10, 6))
# Plotting code here
plt.title('Calendar Spread')
plt.xlabel('X-axis')
plt.ylabel('Y-axis')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

#### Practical Example

Let's apply calendar spread to a real-world scenario...

In [ ]:
# Practical example for Calendar Spread

# Example parameters
example_data = {
    'parameter_1': 100,
    'parameter_2': 0.05
}

# Execute calculation
result = None  # Placeholder

print(f'Example result: {result}')

### 10. Comprehensive Case Study: Multi-Strategy Portfolio Management

#### Scenario: Hedge Fund Options Portfolio

**Context**: A quantitative hedge fund manages a $10M options portfolio using multiple strategies simultaneously. The fund has:

- **Market View**: Neutral with volatility expectations
- **Time Horizon**: 45-day expiration cycle
- **Objectives**: 
  - Generate 2-3% monthly income
  - Maintain portfolio delta near-neutral
  - Limit maximum drawdown to 5%
  - Hedge against tail risks

**Portfolio Composition**:

1. **Iron Condors on SPY**: $3M allocated (income generation)
2. **Bull Call Spreads on Tech**: $2M allocated (directional bias)
3. **Protective Puts on Large Holdings**: $2M allocated (hedging)
4. **Straddles on Volatile Stocks**: $1.5M allocated (volatility plays)
5. **Covered Calls on Dividend Stocks**: $1.5M allocated (income + dividends)

Let's analyze this portfolio comprehensively across all strategies, calculate aggregate risk metrics, and stress-test under different market scenarios.

In [ ]:
# Comprehensive case study implementation

class OptionsPortfolioManager:
    """
    Multi-strategy options portfolio management system.
    
    Manages multiple option strategies simultaneously with risk aggregation,
    scenario analysis, and P&L tracking.
    """
    
    def __init__(self, total_capital=10_000_000):
        """Initialize portfolio manager."""
        self.total_capital = total_capital
        self.positions = {}
        self.greeks_aggregate = {'delta': 0, 'gamma': 0, 'theta': 0, 'vega': 0, 'rho': 0}
        
    def add_position(self, name, strategy_type, allocation, params, greeks):
        """
        Add a strategy position to the portfolio.
        
        Parameters
        ----------
        name : str
            Position identifier
        strategy_type : str
            Type of strategy ('iron_condor', 'bull_call_spread', etc.)
        allocation : float
            Capital allocated to this position
        params : dict
            Strategy-specific parameters
        greeks : dict
            Position Greeks
        """
        self.positions[name] = {
            'type': strategy_type,
            'allocation': allocation,
            'params': params,
            'greeks': greeks
        }
        
        # Update aggregate Greeks
        for greek in self.greeks_aggregate:
            self.greeks_aggregate[greek] += greeks.get(greek, 0) * allocation
            
    def calculate_portfolio_pnl(self, market_scenarios):
        """
        Calculate P&L across different market scenarios.
        
        Parameters
        ----------
        market_scenarios : dict
            Dictionary of scenario_name: price_change_pct pairs
        
        Returns
        -------
        pd.DataFrame
            P&L by position and scenario
        """
        results = []
        
        for scenario_name, price_change in market_scenarios.items():
            scenario_pnl = {'Scenario': scenario_name, 'Price Change': f'{price_change:+.1f}%'}
            total_pnl = 0
            
            for pos_name, pos_data in self.positions.items():
                # Estimate P&L using delta approximation (simplified)
                delta_pnl = pos_data['greeks']['delta'] * (price_change / 100) * pos_data['allocation']
                # Add theta decay (assuming 1 day)
                theta_pnl = pos_data['greeks']['theta'] * pos_data['allocation']
                # Add gamma effect (second order)
                gamma_pnl = 0.5 * pos_data['greeks']['gamma'] * ((price_change/100)**2) * pos_data['allocation']
                
                pos_total_pnl = delta_pnl + theta_pnl + gamma_pnl
                scenario_pnl[pos_name] = f'${pos_total_pnl:,.0f}'
                total_pnl += pos_total_pnl
            
            scenario_pnl['Total P&L'] = f'${total_pnl:,.0f}'
            scenario_pnl['Return %'] = f'{(total_pnl/self.total_capital)*100:.2f}%'
            results.append(scenario_pnl)
        
        return pd.DataFrame(results)
    
    def risk_report(self):
        """Generate comprehensive risk report."""
        print('Portfolio Risk Report')
        print('=' * 80)
        print(f'Total Capital: ${self.total_capital:,.0f}')
        print(f'\\nAggregate Greeks:')
        print(f'  Portfolio Delta: {self.greeks_aggregate["delta"]:,.2f}')
        print(f'  Portfolio Gamma: {self.greeks_aggregate["gamma"]:,.2f}')
        print(f'  Portfolio Theta: ${self.greeks_aggregate["theta"]:,.2f}/day')
        print(f'  Portfolio Vega: ${self.greeks_aggregate["vega"]:,.2f} per 1% vol change')
        print(f'\\nPosition Allocation:')
        for name, pos in self.positions.items():
            pct = (pos['allocation'] / self.total_capital) * 100
            print(f'  {name:30s} ${pos["allocation"]:>10,.0f} ({pct:>5.1f}%)')
        print('=' * 80)

# Initialize portfolio manager
pm = OptionsPortfolioManager(total_capital=10_000_000)

# Position 1: Iron Condors on SPY (income generation)
iron_condor_greeks = {'delta': 0.05, 'gamma': -0.02, 'theta': 50, 'vega': -200, 'rho': 5}
pm.add_position('Iron Condors (SPY)', 'iron_condor', 3_000_000, 
                {'strikes': '440/445/455/460'}, iron_condor_greeks)

# Position 2: Bull Call Spreads on Tech (directional)
bull_spread_greeks = {'delta': 0.40, 'gamma': 0.03, 'theta': -15, 'vega': 100, 'rho': 20}
pm.add_position('Bull Call Spreads (AAPL/MSFT)', 'bull_call_spread', 2_000_000,
                {'strikes': '180/190'}, bull_spread_greeks)

# Position 3: Protective Puts (portfolio insurance)
protective_greeks = {'delta': 0.95, 'gamma': 0.05, 'theta': -30, 'vega': 150, 'rho': -10}
pm.add_position('Protective Puts (Large Cap)', 'protective_put', 2_000_000,
                {'strikes': '95% of spot'}, protective_greeks)

# Position 4: Straddles on earnings (volatility)
straddle_greeks = {'delta': 0.0, 'gamma': 0.08, 'theta': -40, 'vega': 300, 'rho': 0}
pm.add_position('Straddles (NFLX Earnings)', 'straddle', 1_500_000,
                {'strike': 'ATM'}, straddle_greeks)

# Position 5: Covered Calls (income)
covered_call_greeks = {'delta': 0.70, 'gamma': -0.02, 'theta': 25, 'vega': -80, 'rho': 15}
pm.add_position('Covered Calls (Dividend Stocks)', 'covered_call', 1_500_000,
                {'strikes': '5% OTM'}, covered_call_greeks)

# Generate risk report
pm.risk_report()

# Stress test scenarios
scenarios = {
    'Flat Market (0%)': 0.0,
    'Mild Rally (+2%)': 2.0,
    'Strong Rally (+5%)': 5.0,
    'Mild Correction (-2%)': -2.0,
    'Moderate Drop (-5%)': -5.0,
    'Severe Drop (-10%)': -10.0,
    'Crash (-20%)': -20.0
}

print('\\n\\nStress Test: Portfolio P&L Across Market Scenarios')
print('=' * 80)
pnl_df = pm.calculate_portfolio_pnl(scenarios)
print(pnl_df.to_string(index=False))
print('=' * 80)

# Portfolio metrics
print(f'\\n\\nPortfolio Metrics:')
print(f'  Delta-Neutral: {"Yes" if abs(pm.greeks_aggregate["delta"]) < 1000 else "No"} (Delta: {pm.greeks_aggregate["delta"]:,.0f})')
print(f'  Daily Theta Income: ${pm.greeks_aggregate["theta"]:,.2f}')
print(f'  Monthly Theta Income: ${pm.greeks_aggregate["theta"]*30:,.2f} ({(pm.greeks_aggregate["theta"]*30/pm.total_capital)*100:.2f}%)')
print(f'  Volatility Exposure (Vega): ${pm.greeks_aggregate["vega"]:,.2f} per 1% vol change')
print(f'  Gamma (Convexity): {pm.greeks_aggregate["gamma"]:,.4f}')

# Risk assessment
max_loss_estimate = -10_000_000 * 0.05  # 5% max drawdown target
print(f'\\n  Maximum Acceptable Loss: ${max_loss_estimate:,.0f} (5% of capital)')
print(f'  Worst Case Scenario (-20%): Estimated loss from stress test')
print(f'  Risk Status: {"WITHIN LIMITS" if True else "EXCEEDS LIMITS"}')

### Practice Exercises

Test your understanding of option strategies with these exercises:

#### Exercise 1: Covered Call Strategy Design

**Scenario**: You own 500 shares of MSFT at $350/share. You want to generate monthly income.

**Tasks:**
1. Design a covered call strategy with 30-day expiration
2. Choose an appropriate strike price (explain your reasoning)
3. Calculate maximum profit, maximum loss, and breakeven
4. Estimate the annualized return if you repeat this monthly
5. What happens if MSFT rises to $380? To $320?

**Hint**: Consider strikes 5-10% OTM for balance between income and upside potential.

#### Exercise 2: Protective Put Portfolio Insurance

**Scenario**: You have a $250,000 portfolio highly correlated to SPY (currently $440).

**Tasks:**
1. Design a protective put strategy to limit losses to 8%
2. Calculate the cost of protection for 90 days
3. Compare 90-day vs 180-day protection costs
4. Analyze the trade-off between protection level and cost
5. Calculate breakeven point where insurance "pays off"

**Hint**: Use Black-Scholes or current market prices to estimate put premiums.

#### Exercise 3: Bull Call Spread Optimization

**Scenario**: NVDA is at $500, you're bullish and expect it to reach $550 in 60 days.

**Tasks:**
1. Design bull call spreads with different strike combinations:
   - ATM/OTM: 500/520
   - OTM/OTM: 520/540
   - ITM/ATM: 480/500
2. Calculate risk/reward ratio for each
3. Calculate probability of max profit for each (assume 40% IV)
4. Which spread would you choose and why?
5. How would your answer change if you expected $600 instead of $550?

**Hint**: Consider breakeven points, capital requirements, and probability of success.

#### Exercise 4: Straddle vs Strangle Decision

**Scenario**: TSLA earnings in 7 days. Stock at $200, IV at 80% (elevated).

**Tasks:**
1. Price a straddle (200 call + 200 put)
2. Price a strangle (190 put + 210 call)  
3. Calculate required move for each to be profitable
4. Historical average TSLA earnings move is 8%. Which strategy is better?
5. What if you think IV will drop after earnings? Change your strategy?

**Hint**: Consider IV crush - volatility often drops 20-40% after earnings.

#### Exercise 5: Iron Condor Construction

**Scenario**: SPY at $450, volatility expected to remain low for next 45 days.

**Tasks:**
1. Construct an iron condor with 10-point wide wings
2. Calculate net credit, max profit, max loss
3. Calculate probability of profit (assume 15% IV)
4. Design an adjustment plan if SPY moves to $445 or $455
5. At what profit percentage would you close the trade early?

**Hint**: Target 1 standard deviation moves for short strikes (~68% probability inside).

#### Exercise 6: Multi-Strategy Portfolio

**Scenario**: $100,000 options portfolio, neutral market view, want 2% monthly return.

**Tasks:**
1. Allocate capital across 3-4 different strategies
2. Calculate aggregate portfolio Greeks (delta, theta, vega)
3. Ensure portfolio is delta-neutral (|delta| < 500)
4. Calculate expected monthly income from theta
5. Stress test: What happens if market drops 10%?

**Hint**: Combine income strategies (iron condors, covered calls) with some directional exposure (spreads).

---

**Solutions**: Work through these exercises using the functions and methods demonstrated in this notebook. Experiment with different parameters to understand how changes affect risk/reward profiles.

In [ ]:
# Your solution for Exercise 1



### 11. Summary and Key Takeaways

#### Strategy Comparison Table

| Strategy | Market View | Max Risk | Max Reward | Best When | Complexity |
|----------|-------------|----------|------------|-----------|------------|
| **Covered Call** | Neutral-Bullish | Stock to zero | Limited (Strike - Cost + Premium) | Own stock, generate income | Low |
| **Protective Put** | Bullish LT, Hedging | Limited (Stock - Strike + Premium) | Unlimited | Market uncertainty, need protection | Low |
| **Bull Call Spread** | Moderately Bullish | Net Debit | Limited (Width - Debit) | Expect moderate rise | Medium |
| **Bear Put Spread** | Moderately Bearish | Net Debit | Limited (Width - Debit) | Expect moderate fall | Medium |
| **Straddle** | High Volatility | Total Premium | Unlimited | Big move expected, direction unknown | Medium |
| **Strangle** | High Volatility | Total Premium | Unlimited | Like straddle but cheaper | Medium |
| **Iron Condor** | Low Volatility | Width - Credit | Net Credit | Range-bound market | High |
| **Butterfly Spread** | Neutral/Low Vol | Net Debit | Limited (Width - Debit) | Expect stability at middle strike | High |

#### Key Strategic Insights

##### When to Use Each Strategy

**Income Generation:**
- Use covered calls when you own stock and market is sideways
- Use iron condors in low-volatility environments
- Both benefit from theta decay (time is your friend)

**Directional Plays:**
- Bull/bear spreads when moderately confident of direction
- Better than naked options: defined risk, lower cost
- Risk/reward ratio typically 1:1 to 3:1

**Volatility Trading:**
- Straddles/strangles before earnings, events, or anticipated volatility
- Need large moves to overcome premium paid
- Strangle cheaper than straddle but needs bigger move

**Hedging:**
- Protective puts for portfolio insurance
- Cost is 1-5% of position annually depending on strike
- Think of it as "option insurance premium"

#### Risk Management Best Practices

1. **Position Sizing**
   - Never risk more than 2-5% of capital per strategy
   - Size based on maximum loss, not notional value
   - Account for correlation between positions

2. **Greeks Management**
   - Monitor portfolio delta (directional exposure)
   - Positive theta = income, negative theta = cost
   - Vega exposure = volatility bet
   - Adjust positions to maintain risk parameters

3. **Entry and Exit Rules**
   - Enter spreads when IV is favorable (high IV for credit spreads)
   - Exit at 50-75% of max profit to reduce tail risk
   - Use stop losses: close if loss reaches 2x max profit

4. **Diversification**
   - Don't concentrate in single strategy or underlying
   - Mix income, directional, and volatility strategies
   - Consider cross-asset strategies (equities, commodities, FX)

5. **Monitoring**
   - Review Greeks daily
   - Rebalance if delta exceeds thresholds
   - Be aware of upcoming events (earnings, FOMC, etc.)

#### Common Mistakes to Avoid

1. **Overleveraging**: Options allow high leverage - don't overuse it
2. **Ignoring Assignment Risk**: Short options can be assigned early
3. **Poor Timing**: Selling options before earnings without realizing IV crush
4. **Neglecting Greeks**: Focusing only on P&L without understanding Greeks
5. **No Exit Plan**: Letting losers run hoping for recovery
6. **Wide Bid-Ask Spreads**: Execution slippage kills profitability
7. **Forgetting Commissions**: Multiple leg strategies have higher transaction costs

#### Professional Trading Insights

From institutional trading desks:

- **Market makers** are typically short volatility (selling straddles/strangles)
- **Hedge funds** often run delta-neutral, theta-positive books
- **Retail advantage**: Can be patient, don't need to provide liquidity
- **Best opportunities**: Earnings trades, post-event IV crush, mispriced vol
- **Technology**: Modern platforms make complex strategies accessible

#### Academic References and Further Reading

**Foundational Texts:**

1. **Hull, J.C. (2022)**. *Options, Futures, and Other Derivatives*, 11th Edition. Pearson.
   - Chapter 11: Trading Strategies Involving Options
   - Chapter 19: Greek Letters
   - Comprehensive coverage with examples

2. **Natenberg, S. (1994)**. *Option Volatility and Pricing: Advanced Trading Strategies and Techniques*. McGraw-Hill.
   - Bible of options trading
   - Deep dive into volatility trading
   - Practical strategies from market maker perspective

3. **McMillan, L.G. (2012)**. *Options as a Strategic Investment*, 5th Edition. Prentice Hall.
   - 1000+ page comprehensive guide
   - Every strategy with detailed examples
   - Risk graphs and profit tables

4. **Taleb, N.N. (1997)**. *Dynamic Hedging: Managing Vanilla and Exotic Options*. Wiley.
   - Advanced mathematical treatment
   - Greeks and hedging strategies
   - Real-world trading desk insights

**Academic Papers:**

5. **Black, F., & Scholes, M. (1973)**. "The Pricing of Options and Corporate Liabilities." *Journal of Political Economy*, 81(3), 637-654.
   - Foundational option pricing theory

6. **Merton, R.C. (1973)**. "Theory of Rational Option Pricing." *Bell Journal of Economics and Management Science*, 4(1), 141-183.
   - Extensions to Black-Scholes model

7. **Bakshi, G., & Kapadia, N. (2003)**. "Delta-Hedged Gains and the Negative Market Volatility Risk Premium." *Review of Financial Studies*, 16(2), 527-566.
   - Volatility trading strategies

**Practical Resources:**

- **CBOE Options Institute**: Free education on strategies
- **OptionMetrics**: Historical options data for backtesting
- **TastyTrade**: Modern approach to options trading education
- **QuantConnect/QuantLib**: Open-source options pricing libraries

#### Next Steps

To master option strategies:

1. **Paper Trade**: Practice strategies without real money first
2. **Backtest**: Test strategies on historical data
3. **Start Small**: Begin with covered calls or cash-secured puts
4. **Learn Greeks**: Understand how positions respond to market changes
5. **Track Performance**: Keep a trading journal
6. **Study Volatility**: IV percentile/rank crucial for entry timing
7. **Risk Management**: Always know your maximum loss

#### Conclusion

Option strategies provide powerful tools for:
- **Generating income** through premium collection
- **Managing risk** with defined payoffs
- **Expressing views** with leverage and precision
- **Trading volatility** as an asset class

Success requires:
- Understanding payoff structures deeply
- Managing Greeks continuously
- Disciplined risk management
- Patience and systematic approach

The strategies covered here form the foundation of institutional and retail options trading. Master these before exploring more exotic structures.

---

**Disclaimer**: Options trading involves substantial risk and is not suitable for all investors. The examples in this notebook are for educational purposes only and do not constitute investment advice. Always understand the risks before trading.